# UK Biobank: Non-Academic Impact

This is the sole active notebook for analysis 04. It preserves the complete clinical-
trial, patent, policy/Altmetric, collaboration, and publication-figure workflows that
previously occupied five separately executed notebooks.

| Part | Retained source | Role |
|---:|---|---|
| I | `04_non_academic_01_clinical_trials.ipynb` | Linked clinical trials: diseases, status, recruitment, geography, and cited UK Biobank papers |
| II | `04_non_academic_02_patents.ipynb` | Patent geography, topics, legal status, networks, and exports |
| III | `04_non_academic_03_altmetric.ipynb` | Policy and Altmetric attention linked to eligible corpus papers |
| IV | `04_non_academic_04_collaboration.ipynb` | Non-academic collaborators by sector, institution, year, discipline, and geography |
| V | `04_non_academic_99_all.ipynb` | Harmonised main-paper and supplementary panel assembly |

The parts run in that dependency order. Each execution boundary resets the Python
namespace, warnings, open figures, and Matplotlib defaults to reproduce the clean kernel
that separated the former notebooks; files written by earlier parts remain available to
later parts. The project-wide analysis window remains 2013–2025 inclusive.


# Part I: Clinical-trial impact

How UK Biobank research reaches into **clinical trials**. This part drills into the
clinical trials that Dimensions links to UK Biobank in the current frozen source, with one framing point kept front of
mind for the write-up:

> **These are not trials *run by* UK Biobank.** They are clinical trials whose registry
> record **references UKB-related papers** — *every linked trial cites at least
> one paper in our UKB corpus* (§8). So the story is **influence / impact**: UKB findings
> are informing the design and rationale of real-world trials, and that pipeline is a case
> for continued / expanded funding.

**Sections**
1. **Setup** — imports, environment, the shared clinical-trials `STYLE` dict
2. **Data loading, cleaning & missing-data report** — CT-specific parsing (diseases, org
   sectors, enrollment, referenced papers) + a full missing report
3. **What — diseases studied** — MeSH (as shipped, and **rebuilt** from the CTgov
   API as condition-**leaf** terms), free-text `conditions` (word cloud), and **RCDC**
   categories; all three axes mapped to **ICD-10 chapters**
4. **Status** — aggregated trial lifecycle stage, by study type
5. **Recruitment size** — planned enrollment per trial (`study_participants`)
6. **When** — trials by start year
7. **Who / where** — organisations by country **and sector** (academia vs industry vs …)
8. **The UKB papers behind the trials** — match `publication_ids` to `df_dimensions`,
   persist the matched set, and explore what those high-impact papers are about
9. **Combined panel** — the headline views in one figure

> **Modularisation note.** Shared loaders and reusable plotting helpers live under
> `src/utils`; section-specific diagnostics remain here.

## 1. Setup

Self-contained setup — imports, working directory, and the shared
formatting dictionary `STYLE` (four-colour palette, sizes, dpi) that every figure reads.

In [ ]:
import os
import sys
import re
import ast
import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

# Make `src` importable and anchor all paths on the repo root, regardless of the
# directory this notebook is launched from (repo root, src/, or src/data_analysis/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, filter_analysis_window

# Reuse the patent country matcher (pycountry + fuzzy) for the world map.
from utils import shared_patent_utils as patent


In [ ]:
# =============================================================================
# SHARED FORMATTING DICTIONARY  -- every figure reads from this
# =============================================================================
# The palette, figure sizes, font sizes, semantic colour roles and output directory
# live in universal_settings.yml at the repo root, section style.notebooks.04_non_academic_01_clinical_trials.
# load_style merges the shared `base` with this notebook's overrides, resolves the
# semantic roles (c_accent, c_gold, c_green, c_primary) against the palette, registers the result for
# savefig()/extended_palette() and pushes it into rcParams — everything the STYLE
# literal used to do here, minus the drift between notebooks.
from utils.shared_style import apply_style, extended_palette, load_style, savefig, use_style
from utils.shared_style import apply_typography, set_title
apply_typography()
STYLE = load_style("04_non_academic_01_clinical_trials")

print("STYLE ready — palette:", STYLE["colors"])


## 2. Data loading, cleaning & missing-data report

The parsing helpers (`parse_listcol`, `parse_dictcol`, `item_names`, `count_items`,
`split_for_levels`, `add_for_columns`, `missing_report`) are **the same rules** as
the shared source loader. On top of them this part adds four CT-specific derivations:

* **diseases** — `mesh_terms` (controlled vocabulary) and `conditions` (free text)
* **organisation sector** — each research org's Dimensions `types` mapped to a sector
  (Academia / Healthcare / Industry / Government / Nonprofit / Other)
* **recruitment size** — `study_participants` (planned enrollment)
* **referenced papers** — `publication_ids` (the UKB papers each trial cites)

In [ ]:
from utils.shared_showcase import (parse_listcol, parse_dictcol, item_names,
                                   count_items, count_items_frac)
# =============================================================================
# PARSING HELPERS — verbatim from ..._00_all.ipynb (same cleaning rules)
# =============================================================================
# ---- Fields of Research (FOR, 2020 ANZSRC) L2/L4 splitter -------------------
# Shared implementation: src/utils/for_utils.py. It resolves the FOR column itself —
# newer Dimensions exports renamed `category_for_2020` -> `category_for` (same taxonomy),
# and the old local copy SILENTLY returned the frame unchanged when the column was
# missing, producing no for_l2/for_l4 at all. The shared version raises instead.
from utils.shared_for import split_for_levels, add_for_columns, resolve_for_column


def _col_kind(series):
    for v in series.dropna():
        if isinstance(v, list):
            return "list"
        if isinstance(v, dict):
            return "dict"
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("["):
                return "list"
            if s.startswith("{"):
                return "dict"
        return "flat"
    return "empty"


def missing_report(df, cols=None):
    n = len(df)
    cols = list(df.columns) if cols is None else cols
    rows = []
    for c in cols:
        kind = _col_kind(df[c])
        miss = int(df[c].isna().sum())
        if kind == "list":
            ne = int(df[c].apply(lambda x: len(parse_listcol(x)) > 0).sum())
        elif kind == "dict":
            ne = int(df[c].apply(lambda x: len(parse_dictcol(x)) > 0).sum())
        else:
            ne = n - miss
        rows.append((c, kind, miss, round(100 * miss / n, 1), ne, round(100 * ne / n, 1)))
    return (pd.DataFrame(rows, columns=["column", "kind", "n_missing", "%_missing",
                                        "n_nonempty", "%_nonempty"])
            .sort_values("%_missing", ascending=False, ignore_index=True))

In [ ]:
# =============================================================================
# ORGANISATION SECTOR — Dimensions `types` first, then infer the untyped from the name
# =============================================================================
# research_orgs entries carry a `types` list, e.g. ['Education'], ['Company'], ['Healthcare'].
# Two steps, because ~5% of orgs come through with a non-committal GRID type
# ('Facility', 'Other') or no type at all — and those were nearly all universities,
# companies and hospitals that GRID simply hadn't classified (see the audit: 28 'Other'
# orgs, of which only 2 were genuinely ambiguous).
#
#   step 1  trust the CLEAR Dimensions types (Education/Company/Healthcare/Government/Nonprofit)
#   step 2  for 'Facility' / 'Other' / untyped, INFER the sector from the org name; only
#           orgs that match nothing stay 'Other'
#
# The two sectors the impact story cares about most are ACADEMIA and INDUSTRY.
import re

# step 1: the unambiguous Dimensions types (note: 'Facility'/'Other' are NOT here — they
# fall through to name inference rather than being dumped into an 'Other' bucket).
ORG_TYPE_TO_SECTOR = {
    "Education":  "Academia",
    "Company":    "Industry",
    "Healthcare": "Healthcare",
    "Government": "Government",
    "Nonprofit":  "Nonprofit",
}

# step 2: name -> sector, tried IN ORDER (first match wins). Order matters:
#   * Industry / Government are the most specific signals, so first;
#   * Healthcare (hospital/clinic) BEFORE Academia, so a teaching hospital
#     ('University Hospital of X', 'Centre Hospitalier Universitaire') counts as care
#     delivery rather than as a university;
#   * the research-institute rule has no trailing \b — accented characters
#     ('Investigación') are word-chars, so \b would fail mid-word.
ORG_NAME_RULES = [
    ("Industry",   r"\b(inc|corp|corporation|ltd|llc|gmbh|pharma\w*|therapeutics|biologics|"
                   r"biosciences|technologies|diagnostics|laboratories)\b"),
    ("Government", r"\b(ministry|statens|national institutes? of health|public health|"
                   r"serum institut|centers? for disease|agency)\b"),
    ("Healthcare", r"\b(hospital|hospitalier|medical cent(er|re)|clinic|health system|"
                   r"infirmary|nhs|clinical research facility)\b"),
    ("Academia",   r"\b(univer\w+|college|school of medicine|institute of technology|"
                   r"polytechnic|université|academ\w+)\b"),
    ("Academia",   r"\b(research institut|research cent|clinical research|biomedical research|"
                   r"institut|fondazione|centro de investiga|biocenter|foundation)"),
]


def infer_sector_from_name(name):
    """Sector guessed from an org name when Dimensions gave no usable type; else 'Other'."""
    n = (name or "").lower()
    for sector, pattern in ORG_NAME_RULES:
        if re.search(pattern, n):
            return sector
    return "Other"


def org_sector(org):
    """Sector for one research-org dict: a clear Dimensions type wins; otherwise infer from
    the name (so 'Facility'/'Other'/untyped orgs are placed rather than lumped as 'Other')."""
    for t in (org.get("types") or []):
        if t in ORG_TYPE_TO_SECTOR:
            return ORG_TYPE_TO_SECTOR[t]
    return infer_sector_from_name(org.get("name"))


SECTOR_ORDER  = ["Academia", "Healthcare", "Industry", "Government", "Nonprofit", "Other"]
SECTOR_COLORS = {
    "Academia":   STYLE["c_primary"],   # blue
    "Industry":   STYLE["c_accent"],    # red
    "Healthcare": STYLE["c_green"],     # green
    "Government": STYLE["c_gold"],      # gold
    "Nonprofit":  "#7B6D8D",            # muted purple
    "Other":      "#B8B8B8",            # grey
}

In [ ]:
# see what's the other here? 
# some other tags print them into the text 



In [ ]:
# =============================================================================
# LOAD the clinical-trials data + CT-specific derivations
# =============================================================================
# SOURCE REPOINTED 2026-09-22, the move D43 made for patents. This read the side CSV at
# data/analysis/non_academic/clinical_trials/clinical_trials.csv, which `ensure_*_csv`
# returns whenever it happens to exist — so the arm's index date was a property of the
# filesystem, and that CSV pre-dates the corpus file by three weeks. The records now come
# out of the corpus's OWN `clinical_trials__*` block, the same extraction the publication
# rows come from, through `shared_showcase.endpoint_records` (D44).
#
# It changes no number here: both hold the same 195 trials and agree cell for cell on all
# 46 shared columns. What it removes is the way they could stop agreeing unnoticed. The one
# field the corpus does not carry is `mesh_leaf_ids` (§3.5.2's MeSH-descriptor D-numbers,
# written by this repo's own ct.gov re-fetch); it is merged back from the CSV by
# `clinical_trials_records`, so §3.5.2 is unaffected.
#
# List/dict columns now arrive as real Python objects rather than JSON strings.
# `parse_listcol` keeps an already-parsed list as-is, so every use below is unchanged.
from utils.data_analysis_04_non_academic_sources import clinical_trials_records
df_ct = filter_analysis_window(
    clinical_trials_records(), year_col="start_year", date_col="start_date"
)
print(f"Clinical trials : {df_ct.shape[0]} rows x {df_ct.shape[1]} cols")

# FOR L2/L4 (same splitter as _00_all)
add_for_columns(df_ct, "category_for_2020")

# WHEN — start_year (0% missing) + active_years as a real list
df_ct["start_date_parsed"] = pd.to_datetime(df_ct["start_date"], errors="coerce")
df_ct["start_year"] = df_ct["start_date_parsed"].dt.year
df_ct["active_years_list"] = df_ct["active_years"].apply(parse_listcol)

# REFERENCED PAPERS — publication_ids as a list; these link each trial to UKB papers (§8)
df_ct["pids"] = df_ct["publication_ids"].apply(parse_listcol)
df_ct["n_pids"] = df_ct["pids"].apply(len)

# ORG SECTORS present in each trial (list, deduped) — for the who/where + sector view
def _trial_sectors(cell):
    return sorted({org_sector(o) for o in parse_listcol(cell) if isinstance(o, dict)})
df_ct["org_sectors"] = df_ct["research_orgs"].apply(_trial_sectors)

# quick coverage prints
_has_country = df_ct["research_orgs"].apply(
    lambda x: any(isinstance(d, dict) and d.get("country_name") for d in parse_listcol(x)))
print(f"Trials with >=1 research-org country : {_has_country.sum()}/{len(df_ct)}")
print(f"Trials referencing >=1 paper (pids)  : {(df_ct['n_pids']>0).sum()}/{len(df_ct)} "
      f"(mean {df_ct['n_pids'].mean():.1f} pids/trial)")
print(f"Trials involving >=1 Industry org    : {df_ct['org_sectors'].apply(lambda s: 'Industry' in s).sum()}")
print(f"Trials involving >=1 Academia org    : {df_ct['org_sectors'].apply(lambda s: 'Academia' in s).sum()}")
print(f"Start years : {int(df_ct['start_year'].min())}–{int(df_ct['start_year'].max())}")
print(f"Enrollment (study_participants): median {int(df_ct['study_participants'].median())}, "
      f"max {int(df_ct['study_participants'].max())}, total {int(df_ct['study_participants'].sum()):,}")

In [ ]:
df_ct_interventional = df_ct[df_ct['study_type'] == 'Interventional'].copy()

df_ct_observational = df_ct[df_ct['study_type'] == 'Observational'].copy()

df_ct_interventional.to_csv(P.CLINICAL_TRIALS / "clinical_trials_interventional.csv", index=False)
df_ct_observational.to_csv(P.CLINICAL_TRIALS / "clinical_trials_observational.csv", index=False)


In [ ]:
df_ct_interventional

In [ ]:
df_ct_observational

### Missing-data report (quantified, every column)

Automated report for the clinical-trials file only.

In [ ]:
print(f"CLINICAL TRIALS  (n = {len(df_ct)}, {df_ct.shape[1]} columns)")
with pd.option_context("display.max_rows", None):
    display(missing_report(df_ct))

**Data-handling decisions (CT-specific).**

* **Diseases** — two complementary columns. `mesh_terms` (8% missing) is a **controlled
  vocabulary** → used for the ranked bar, shown **raw and fractional**. `conditions` (0%
  missing) is **free text** the sponsor typed → used for a **word cloud** (too many unique
  phrasings for a bar). Fractional counting spreads a weight of 1 across each trial's
  diseases so a trial tagged with 8 MeSH terms doesn't outweigh one tagged with 1.
* **ICD codes** — checked: the CT export carries **no ICD field**, and neither `conditions`
  nor `mesh_terms` contains ICD codes (MeSH is a *different* controlled vocabulary). A
  MeSH→ICD-10 crosswalk exists only in external resources (UMLS Metathesaurus) and can't be
  done reliably offline, so we treat **MeSH as the disease axis** and do not fabricate ICD.
* **Status** — analysed only at the **aggregated lifecycle stage** (see §4 for the exact
  9→5 mapping), split by interventional vs. observational `study_type`.
* **Recruitment size** — `study_participants` is the **planned/target** enrollment from the
  registry (0% missing, 1 zero). It is heavily right-skewed (median ≈ 180, max 150,000), so
  it is shown on a **log** axis and bucketed.
* **Who/where** — `research_orgs` (4% missing) gives country **and** `types`→sector. Each
  trial's (country, sector) pairs are counted **fractionally** so the stacked bar sums to
  the number of located trials. **These orgs run the trials that cite UKB — UKB is not itself
  the sponsor.**
* **Referenced papers** — `publication_ids` (0% missing, mean 23 pids/trial) are matched
  against `df_dimensions` in §8; the intersection is the set of **UKB papers** the trials cite.

### MeSH terms (controlled vocabulary)

In [ ]:
df_ct.loc[df_ct['mesh_terms'].isnull()]

In [ ]:
[c for c in df_ct.columns if 'mesh' in c]

## 3. What — diseases studied

Two views of the same question, from the two disease columns.

In [ ]:
import textwrap
# ---- Reusable STYLE-driven primitives ---------------------------------------
def plot_hbar(counts, title, xlabel, style=STYLE, color=None, ax=None,
              annotate=True, pct_of=None, fmt="{:.0f}", name="hbar",ylabel=None):
    """Horizontal bar chart of a pre-sorted count Series (largest on top)."""
    color = color or style["c_primary"]
    counts = counts.sort_values(ascending=True, kind="stable")
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(style["figsize_small"][0],
                                        max(3.5, 0.45 * len(counts) + 1)))
    else:
        fig = ax.figure
    bars = ax.barh(counts.index.astype(str), counts.values,
                   color=color, edgecolor=style["edgecolor"], linewidth=1)
    if annotate:
        for b, v in zip(bars, counts.values):
            lab = fmt.format(v) + (f"  ({100*v/pct_of:.0f}%)" if pct_of else "")
            ax.text(v + max(counts.values) * 0.01, b.get_y() + b.get_height()/2,
                    lab, va="center", fontsize=style["annot_fs"])
    set_title(ax, title, fontsize=style["title_fs"], loc="left", fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=style["label_fs"])
    
    # for all the y-ticklabels, set the textwrap 
    for label in ax.get_yticklabels():
        label.set_text("\n".join(textwrap.wrap(label.get_text(), 20)))
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=style["label_fs"])
    ax.margins(x=0.12)
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax


def plot_wordcloud(freqs, title, style=STYLE, max_words=120, ax=None, name="wordcloud",width=1600, height=800):
    """Word cloud from a {word: weight} mapping (or Series), coloured from the palette."""
    import random
    from wordcloud import WordCloud
    freqs = dict(freqs)
    palette = style["colors"]
    fallback_rng = random.Random(42)
    def _color(*a, random_state=None, **k):
        return (random_state or fallback_rng).choice(palette)
    wc = WordCloud(width=width, height=height, background_color="white", prefer_horizontal=0.95,
                   color_func=_color, max_words=max_words, collocations=False,
                   random_state=42).generate_from_frequencies(freqs)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=style["figsize_wide"])
    else:
        fig = ax.figure
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    set_title(ax, title, fontsize=style["title_fs"], fontweight="bold")
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax

### 3.1 Diseases (MeSH terms) — raw vs. fractional counts

> ⚠️ **Read this bar with §3.5.** `mesh_terms` as shipped by Dimensions merges the MeSH the
> registry matched with its **tree ancestors** and with **intervention**-derived terms, so the
> ranking below is topped by scaffolding (*Pathologic Processes*, *Pathological Conditions,
> Signs and Symptoms*) rather than by diseases. §3.5 rebuilds this from the ClinicalTrials.gov
> API keeping only the **condition-leaf** terms — that is the disease bar to quote.

In [ ]:
mesh_raw  = count_items(df_ct["mesh_terms"], top=15)
mesh_frac = count_items_frac(df_ct["mesh_terms"]).head(15)
n_mesh = int(df_ct["mesh_terms"].apply(lambda x: len(parse_listcol(x)) > 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_hbar(mesh_raw, "a.",
          "Number of trials — most-studied MeSH categories (raw)", color=STYLE["c_primary"],  ax=axes[0])
plot_hbar(mesh_frac, "b.",
          "Fractional trial weight — most-studied MeSH categories", color=STYLE["c_gold"], fmt="{:.1f}", ax=axes[1])
#fig.suptitle(f"Diseases studied in UKB-linked clinical trials "
#             f"({n_mesh}/{len(df_ct)} trials MeSH-coded)",
#             fontsize=STYLE["title_fs"] + 1, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94]); savefig(fig, "ct_diseases_mesh")
plt.show()

### 3.2 Diseases (free-text `conditions`) — normalised word cloud

`conditions` is sponsor free text, so the same disease appears under many surface forms
(*Type 2 diabetes / Diabetes mellitus / Diabetes*; *Atrial fibrillation / Atrial
fibrillation (AF)*). Before the cloud we **normalise** each phrase — lowercase, drop
parenthetical acronyms, then a small synonym map merges the obvious families — and count
**fractionally** (each trial spreads weight 1 across its distinct normalised conditions).
The synonym map is deliberately conservative and easy to extend.

In [ ]:
import html

# Conservative synonym map. KEYS are in the *normalised* form norm_key() produces
# (lowercase, no parentheticals, commas/&/slashes -> spaces, collapsed) so lookups hit.
COND_SYNONYMS = {}
def _syn(canonical, *variants):
    for v in variants:
        COND_SYNONYMS[v] = canonical

_syn("Diabetes", "diabetes", "type 2 diabetes", "type 1 diabetes", "diabetes mellitus",
     "diabetes mellitus type 2", "type 2 diabetes mellitus", "diabetes mellitus type 1",
     "type 1 diabetes mellitus", "t2dm", "niddm", "diabetes autoimmune")
_syn("Obesity", "obesity", "obesity morbid", "morbid obesity", "obesity adolescent",
     "adolescent obesity", "obesity childhood", "childhood obesity", "overweight",
     "overweight and obesity", "obesity overweight", "overweight childhood",
     "adolescent overweight")
_syn("Cardiovascular disease", "cardiovascular disease", "cardiovascular diseases",
     "cardiovascular risk", "cardiovascular diseases risk")
_syn("Coronary heart disease", "coronary heart disease", "coronary artery disease")
_syn("Atrial fibrillation", "atrial fibrillation")
_syn("Alzheimer's disease", "alzheimer's disease", "alzheimer disease", "alzheimers disease")
_CANON = set(COND_SYNONYMS.values())


def norm_key(name):
    """Normalise a free-text condition to a merge key (unescape HTML, lowercase, drop
    '(...)', turn commas/&/slashes into spaces, collapse), then apply the synonym map."""
    s = html.unescape(str(name)).lower()
    s = re.sub(r"\(.*?\)", " ", s)          # drop parenthetical acronyms e.g. (AF)
    s = re.sub(r"[.,;/&]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return COND_SYNONYMS.get(s, s)


def normalized_condition_freq(series):
    """Fractional frequency of NORMALISED conditions. Each trial spreads weight 1 across
    its distinct merge-keys; the display label is the synonym canonical (if any) else the
    most common original surface form for that key."""
    kw = defaultdict(float)
    surf = defaultdict(Counter)
    for cell in series:
        keys = set()
        for it in item_names(cell):
            k = norm_key(it)
            keys.add(k); surf[k][html.unescape(it)] += 1
        if not keys:
            continue
        f = 1.0 / len(keys)
        for k in sorted(keys):
            kw[k] += f
    out = defaultdict(float)
    for k, w in kw.items():
        label = k if k in _CANON else surf[k].most_common(1)[0][0]
        out[label] += w
    return pd.Series(out, dtype="float64").sort_values(ascending=False)


cond_freq = normalized_condition_freq(df_ct["conditions"])
plot_wordcloud(cond_freq,"", #"Conditions studied (free-text, normalised + fractional)",
               name="ct_conditions_wordcloud")
plt.show()
print("Top normalised conditions:",
      ", ".join(f"{k} ({v:.1f})" for k, v in cond_freq.head(12).items()))

### 3.3 Mapping disease names to ICD-10 chapters

The data carries **no ICD field**, but the **names can be mapped** to ICD — e.g. *Diabetes*
→ **E10–E14** (chapter *IV Endocrine, nutritional & metabolic*), *Atrial fibrillation* →
**I48** (chapter *IX Circulatory*). A full term-level MeSH→ICD-10 crosswalk needs the
licence-gated UMLS Metathesaurus, but a **chapter-level** grouping is both reliable and more
useful for a headline: it collapses the long MeSH list into the ~a dozen ICD-10 body-system
chapters. Below we apply a transparent **keyword → ICD-10 chapter** map to each trial's MeSH
terms (first match wins; MeSH qualifiers like *Pathologic Processes* stay *Unmapped*), then
count trials per chapter (a trial can span several).

> This is a **curated approximation**, not an authoritative UMLS mapping — the `ICD_CHAPTERS`
> rules are in the cell and easy to inspect / extend.

In [ ]:
# Keyword -> ICD-10 chapter (checked in order; first hit wins). Circulatory is listed before
# nervous so 'stroke' (I60-I69) lands in circulatory, etc.
ICD_CHAPTERS = [
    ("II  Neoplasms (C00–D48)",
     ["neoplasm", "cancer", "carcinoma", "tumor", "tumour", "lymphoma", "leukemia",
      "leukaemia", "melanoma", "sarcoma", "malignan", "adenoma"]),
    ("IV  Endocrine, nutritional & metabolic (E00–E90)",
     ["diabet", "obes", "overweight", "metaboli", "nutrition", "lipid", "cholesterol",
      "thyroid", "glucose", "insulin", "hyperglyc", "dyslipid", "hyperlipid"]),
    ("IX  Circulatory system (I00–I99)",
     ["cardiovascular", "cardiac", "heart", "coronary", "atrial fibrillation", "arrhythmia",
      "myocard", "vascular", "hypertension", "atheroscler", "thrombo", "ischemi", "ischaemi",
      "stroke", "aneurysm", "angina", "cardiomyopath", "embolism"]),
    ("V   Mental & behavioural (F00–F99)",
     ["depress", "anxiety", "psychiatr", "psychosis", "schizophren", "bipolar", "mood",
      "mental disorder", "substance", "addiction", "alcohol"]),
    ("VI  Nervous system (G00–G99)",
     ["alzheimer", "dementia", "parkinson", "epilep", "migraine", "sclerosis",
      "neurodegener", "cognitive", "neuropath"]),
    ("X   Respiratory system (J00–J99)",
     ["asthma", "copd", "respiratory", "pulmonary disease", "lung disease", "pneumon", "bronch"]),
    ("XI  Digestive system (K00–K93)",
     ["liver", "hepat", "nafld", "gastro", "digest", "bowel", "crohn", "colitis",
      "pancrea", "cirrhosis", "fatty liver", "steato"]),
    ("XIII Musculoskeletal (M00–M99)",
     ["arthr", "osteo", "musculoskeletal", "sarcopenia", "bone", "joint", "rheumat"]),
    ("XIV Genitourinary (N00–N99)",
     ["kidney", "renal", "urinary", "prostat", "nephro", "bladder"]),
    ("XV  Pregnancy & childbirth (O00–O99)",
     ["pregnan", "eclampsia", "obstetric", "gestational"]),
    ("I   Infectious & parasitic (A00–B99)",
     ["infect", "covid", "viral", "bacteri", "sepsis", "hiv"]),
]


def _norm_icd_term(term):
    """Lowercase a term for keyword matching, defusing substring traps.

    'Non-alcoholic Fatty Liver Disease' contains the substring 'alcohol', and the Mental &
    behavioural rules are tested BEFORE Digestive — so every NAFLD trial was landing in
    F00-F99. Stripping the negation makes it match 'liver' -> XI Digestive, which is right
    (NAFLD is K76.0).
    """
    return term.lower().replace("non-alcoholic", " ").replace("nonalcoholic", " ")


def map_icd_chapter(term):
    """First-matching ICD-10 chapter for a disease term, else None (unmapped)."""
    t = _norm_icd_term(term)
    for chapter, kws in ICD_CHAPTERS:
        if any(k in t for k in kws):
            return chapter
    return None


def trials_by_icd_chapter(df, col="mesh_terms"):
    """Trials per ICD-10 chapter (a trial counted once per chapter it touches)."""
    c = Counter()
    n_mapped = 0
    for cell in df[col]:
        chapters = {map_icd_chapter(t) for t in item_names(cell)}
        chapters.discard(None)
        if chapters:
            n_mapped += 1
        c.update(chapters)
    return pd.Series(c, dtype="int64").sort_values(ascending=False), n_mapped


# ---- Human "body map": place each ICD-10 chapter on a body silhouette --------
BODY_COLOR = "#F4C79B"
# full-chapter -> (x, y, short label) anatomical positions; systemic chapters
# (neoplasms, infectious) sit just off the body on the right, like the reference figure.
ICD_BODY_POS = {
    "VI  Nervous system (G00–G99)":                     (5.20, 12, "Nervous"),
    "V   Mental & behavioural (F00–F99)":               (3, 12.0, "Mental"),
    "X   Respiratory system (J00–J99)":                 (4.05, 9.70, "Respiratory"),
    "IX  Circulatory system (I00–I99)":                 (5.70, 9.30, "Circulatory"),
    "IV  Endocrine, nutritional & metabolic (E00–E90)": (5.00, 8.10, "Endocrine/metabolic"),
    "XI  Digestive system (K00–K93)":                   (5.45, 7.05, "Digestive"),
    "XIV Genitourinary (N00–N99)":                      (4.45, 6.05, "Genitourinary"),
    "XV  Pregnancy & childbirth (O00–O99)":             (5.75, 5.75, "Pregnancy"),
    "XIII Musculoskeletal (M00–M99)":                   (4.3, 3.80, "Musculoskeletal"),
    "II  Neoplasms (C00–D48)":                          (8.70, 5, "Neoplasms"),
    "I   Infectious & parasitic (A00–B99)":             (8.70, 4, "Infectious"),
}


def _draw_body(ax, color=BODY_COLOR):
    """Draw a simple pale humanoid silhouette (head, torso, arms, legs)."""
    from matplotlib.patches import FancyBboxPatch, Ellipse
    rr = lambda xy, w, h, s=0.55: FancyBboxPatch(
        xy, w, h, boxstyle=f"round,pad=0.02,rounding_size={s}", fc=color, ec="none")
    for x in (3.75, 5.15):                       # legs (shortened proportions)
        ax.add_patch(rr((x, 2.1), 1.1, 6))
    for x in (2.35, 6.85):                        # arms
        ax.add_patch(rr((x, 6.8), 0.8, 4.1))
    ax.add_patch(rr((3.3, 5.9), 3.4, 5.3, 0.8))  # torso
    ax.add_patch(rr((4.55, 10.9), 0.9, 1.0, 0.2))  # neck
    ax.add_patch(Ellipse((5.0, 12.5), 1.9, 2.2, fc=color, ec="none"))  # head


def plot_icd_bodymap(counts, title, style=STYLE, n_annotate=5, ax=None, name="ct_icd_bodymap"):
    """Body-map of trials per ICD-10 chapter: each chapter is a dot at its anatomical
    position, area proportional to the trial count; the top-`n_annotate` are bold."""
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(8, 9))
    else:
        fig = ax.figure
    _draw_body(ax)
    mx = max(counts.max(), 1)
    top = set(counts.sort_values(ascending=False).head(n_annotate).index)
    for chap, v in counts.items():
        if chap not in ICD_BODY_POS:
            continue
        x, y, short = ICD_BODY_POS[chap]
        ax.scatter([x], [y+0.7], s=140 + (v / mx) * 1500, c=style["c_primary"],
                   edgecolors="#12233F", linewidths=1.2, zorder=5, alpha=0.92)
        if x < 4.7:
            ha, dx = "right", -0.45
        elif x > 7:
            ha, dx = "left", 0.5
        else:
            ha, dx = "left", 0.5
        ax.annotate(f" {short} ({int(v)})", (x, y+0.7), (x + dx, y+0.7), textcoords="data",
                    ha=ha, va="center", fontsize=style["annot_fs"],
                    fontweight="bold" if chap in top else "normal",
                    color="black" if chap in top else "#333333", zorder=6)
    ax.set_xlim(-2.2, 12.2); ax.set_ylim(1.4, 14.2)
    ax.set_aspect("equal"); ax.axis("off")
    set_title(ax, title, fontsize=style["title_fs"], fontweight="bold")
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax


icd_counts, n_icd = trials_by_icd_chapter(df_ct)
plot_icd_bodymap(icd_counts, "Trials by ICD-10 chapter (from MeSH terms)",
                 name="ct_icd_bodymap")
print(f"{n_icd}/{len(df_ct)} trials mapped to >=1 ICD-10 chapter; dot area ∝ trials.")
plt.show()
display(icd_counts.rename("n_trials").to_frame())

### 3.4 Research areas & diseases (RCDC categories)

`category_rcdc` is Dimensions' **RCDC** tagging — NIH's *Research, Condition, and Disease
Categorisation* — and it is **0% missing** (every trial carries a median of 9 tags, 157
distinct tags in total), which makes it a stronger disease axis than `mesh_terms` (8%
missing).

The catch is that RCDC is a **mixed vocabulary**. Next to genuine conditions
(*Cardiovascular*, *Obesity*, *Brain Disorders*) it carries cross-cutting research areas
(*Clinical Research*, *Prevention*), methods (*Biomedical Imaging*), populations (*Women's
Health*) and funding-portfolio labels (*Clinical Trials and Supportive Activities*). Those
sit at the very top of a raw ranking and say nothing about *what disease* a trial studies,
so we keep **two views**:

* **all tags** — what the RCDC portfolio of these trials looks like as tagged;
* **disease-only** — the same counts after a `RCDC_STOP` list drops the cross-cutting tags
  (the list is in the cell below, explicit and easy to edit).

Both are shown **raw and fractional**, exactly as in §3.1 — fractional spreads a weight of 1
across each trial's tags, so a trial carrying 15 RCDC tags doesn't outweigh one carrying 3.

In [ ]:
# =============================================================================
# RCDC (NIH Research, Condition & Disease Categorisation) — tags, stop-list, bars
# =============================================================================
# RCDC mixes diseases with research areas / methods / populations / portfolio labels.
# The stop-list below marks the NON-disease tags; everything else is treated as a condition.
# It is deliberately explicit (not a heuristic) so it can be inspected and extended.
RCDC_STOP = {
    # research-activity / portfolio labels
    "Clinical Research", "Clinical Trials and Supportive Activities", "Prevention",
    "Comparative Effectiveness Research", "Cost Effectiveness Research",
    "Dissemination and Implementation Research", "Patient Safety", "Health Services",
    "Primary Health Care", "Emergency Care", "Telehealth", "Rehabilitation",
    "Physical Rehabilitation", "Precision Medicine", "Orphan Drug", "Transplantation",
    "Organ Transplantation", "Regenerative Medicine",
    # methods / platforms / technology
    "Biomedical Imaging", "Bioengineering", "Biotechnology", "Biodefense",
    "Machine Learning and Artificial Intelligence", "Data Science", "Assistive Technology",
    "Networking and Information Technology R&D (NITRD)",
    # biology / mechanism — not a disease axis
    "Genetics", "Human Genome", "Genetic Testing", "Microbiome", "Neurosciences",
    "Hematology", "Endocannabinoid System Research", "Cannabinoid Research",
    # behavioural / lifestyle / exposure research areas
    "Behavioral and Social Science", "Basic Behavioral and Social Science",
    "Physical Activity", "Dietary Supplements", "Complementary and Integrative Health",
    "Sleep Research", "Breastfeeding, Lactation and Breast Milk", "Contraception/Reproduction",
    # populations / equity / life-stage framings
    "Aging", "Women's Health", "Maternal Health", "Minority Health", "Health Disparities",
    "Health Disparities and Racial or Ethnic Minority Health Research",
    "Social Determinants of Health", "Pediatric Research Initiative", "Rural Health",
    "Rare Diseases", "Infant Mortality", "Vaccine Related", "Immunization",
    "Pain Research",   # the research *field*; the conditions are 'Chronic Pain' / 'Back Pain'
}
RCDC_STOP_PREFIX = ("Stem Cell Research",)   # 5 near-identical variants


def is_disease_rcdc(name):
    """True if an RCDC tag names a condition rather than a research area / method / population."""
    return name not in RCDC_STOP and not name.startswith(RCDC_STOP_PREFIX)


# two list columns: every tag, and the disease-only subset
df_ct["rcdc_all"] = df_ct["category_rcdc"].apply(item_names)
df_ct["rcdc_disease"] = df_ct["rcdc_all"].apply(lambda ts: [t for t in ts if is_disease_rcdc(t)])

n_rcdc     = int((df_ct["rcdc_all"].str.len() > 0).sum())
n_rcdc_dis = int((df_ct["rcdc_disease"].str.len() > 0).sum())
print(f"Trials with >=1 RCDC tag           : {n_rcdc}/{len(df_ct)} "
      f"({count_items(df_ct['rcdc_all']).size} distinct tags, "
      f"median {df_ct['rcdc_all'].str.len().median():.0f} tags per trial)")
print(f"Trials with >=1 *disease* RCDC tag  : {n_rcdc_dis}/{len(df_ct)} "
      f"({count_items(df_ct['rcdc_disease']).size} distinct tags) — "
      f"{len(RCDC_STOP)} cross-cutting tags stop-listed")

# ---- (i) ALL tags — raw vs fractional ---------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_hbar(count_items(df_ct["rcdc_all"], top=15), "a.",
          "Number of trials — all RCDC categories (raw)", color=STYLE["c_primary"], ax=axes[0])
plot_hbar(count_items_frac(df_ct["rcdc_all"]).head(15), "b.",
          "Fractional trial weight — all RCDC categories", color=STYLE["c_gold"], fmt="{:.1f}", ax=axes[1])
fig.tight_layout(w_pad=3); savefig(fig, "ct_rcdc_all")
plt.show()

In [ ]:
# ---- (ii) DISEASE-ONLY tags — raw vs fractional ------------------------------
# The same two counts with the cross-cutting tags removed: this is the RCDC answer to
# "what diseases do the UKB-citing trials study?", directly comparable to the MeSH bars (§3.1).
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_hbar(count_items(df_ct["rcdc_disease"], top=15), "a.",
          "Number of trials — diseases (RCDC, raw)", color=STYLE["c_primary"], ax=axes[0])
plot_hbar(count_items_frac(df_ct["rcdc_disease"]).head(15), "b.",
          "Fractional trial weight — diseases (RCDC)", color=STYLE["c_gold"], fmt="{:.1f}", ax=axes[1])
fig.tight_layout(w_pad=3); savefig(fig, "ct_rcdc_disease")
plt.show()

# the dropped tags, for the record — check nothing disease-like was thrown away
dropped = count_items(df_ct["rcdc_all"].apply(lambda ts: [t for t in ts if not is_disease_rcdc(t)]))
print("Stop-listed (cross-cutting) RCDC tags actually present, by trials carrying them:")
display(dropped.head(15).rename("n_trials").to_frame())

#### 3.4.1 RCDC → ICD-10 chapters

The same **keyword → ICD-10 chapter** idea as §3.3, applied to the RCDC disease tags. Two
changes are needed because RCDC phrases its conditions differently from MeSH (*Brain
Disorders*, *Lung*, *Urologic Diseases*):

* the §3.3 `ICD_CHAPTERS` rules are **extended, not replaced** — RCDC's vocabulary is folded
  into the *same chapter labels*, so a term MeSH would have matched still lands in the same
  chapter and the two mappings stay directly comparable;
* RCDC reaches **six chapters MeSH never did** (blood/immune, eye, skin, perinatal, injury,
  symptoms), appended after the original rules so the first-hit-wins order is preserved.

With those rules **every** disease RCDC tag maps to a chapter, so the RCDC route covers more
trials than the MeSH route (`mesh_terms` is 8% missing and its qualifiers often don't name a
disease). Panel **b** puts the two side by side. No body map here — the chapter bar plus the
MeSH/RCDC comparison is what the write-up needs.

In [ ]:
# =============================================================================
# RCDC -> ICD-10 chapters — extend the §3.3 rules with RCDC's vocabulary
# =============================================================================
# Keywords added to the EXISTING chapters (labels identical to §3.3, so counts are comparable).
RCDC_ICD_EXTRA = {
    "II  Neoplasms (C00–D48)": ["oncolog"],
    "V   Mental & behavioural (F00–F99)": [
        "mental health", "mental illness", "autism", "attention deficit", "adhd",
        "eating disorder", "anorexia", "suicide", "tobacco", "opioid", "drug abuse",
        "intellectual and developmental"],
    "VI  Nervous system (G00–G99)": ["brain disorder", "sleep", "aphasia", "headache"],
    "X   Respiratory system (J00–J99)": ["lung", "influenza"],
    "XI  Digestive system (K00–K93)": ["celiac", "coeliac", "dental", "oral"],
    "XIV Genitourinary (N00–N99)": ["urologic", "infertility", "reproduction"],
    "XV  Pregnancy & childbirth (O00–O99)": ["maternal", "lactation", "breastfeeding"],
    "I   Infectious & parasitic (A00–B99)": ["coronavirus", "vaccine", "immunization"],
}
# Chapters RCDC reaches that the MeSH map never needed — appended, so the §3.3 order still wins.
ICD_CHAPTERS_NEW = [
    ("III Blood & immune (D50–D89)",       ["autoimmune", "anemia", "anaemia", "immune"]),
    ("VII Eye & adnexa (H00–H59)",         ["eye", "vision", "macular", "retin", "glaucoma",
                                            "cataract"]),
    ("XII Skin (L00–L99)",                 ["psoriasis", "skin", "dermat", "eczema"]),
    ("XVI Perinatal conditions (P00–P96)", ["perinatal", "neonat", "embryonic and fetal"]),
    ("XIX Injury & poisoning (S00–T98)",   ["injury", "accident", "adverse effects", "poisoning"]),
    ("XVIII Symptoms & signs (R00–R99)",   ["pain"]),
]


def merge_icd_rules(base=ICD_CHAPTERS, extra=RCDC_ICD_EXTRA, new=ICD_CHAPTERS_NEW):
    """§3.3 chapters in §3.3 order with the RCDC keywords folded in, then the RCDC-only chapters."""
    return [(chap, kws + list(extra.get(chap, []))) for chap, kws in base] + list(new)


ICD_CHAPTERS_RCDC = merge_icd_rules()


def map_icd_chapter_rules(term, rules):
    """First-matching ICD-10 chapter under an explicit rule list, else None (unmapped)."""
    t = _norm_icd_term(term)      # see §3.3: defuses 'non-alcoholic' -> 'alcohol' -> Mental
    for chapter, kws in rules:
        if any(k in t for k in kws):
            return chapter
    return None


def icd_chapter_counts(series, rules=ICD_CHAPTERS_RCDC):
    """Trials per ICD-10 chapter from a list column (a trial counted once per chapter it touches)."""
    c, n_mapped = Counter(), 0
    for cell in series:
        chapters = {map_icd_chapter_rules(t, rules) for t in item_names(cell)}
        chapters.discard(None)
        if chapters:
            n_mapped += 1
        c.update(chapters)
    return pd.Series(c, dtype="int64").sort_values(ascending=False), n_mapped


rcdc_icd, n_rcdc_icd = icd_chapter_counts(df_ct["rcdc_disease"])
mesh_icd, n_mesh_icd = icd_chapter_counts(df_ct["mesh_terms"])       # SAME rules, so the
#                                                                    # comparison is like-for-like:
#                                                                    # only the vocabulary differs

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_hbar(rcdc_icd, "a.", "Number of trials — by ICD-10 chapter (from RCDC)",
          color=STYLE["c_green"], pct_of=len(df_ct), ax=axes[0])

# (b) the two vocabularies over the same chapters
comp = (pd.concat([mesh_icd.rename("MeSH"), rcdc_icd.rename("RCDC")], axis=1)
          .fillna(0).astype(int).sort_values("RCDC"))
ax = axes[1]
y, h = np.arange(len(comp)), 0.4
for i, (src, col) in enumerate([("MeSH", STYLE["c_primary"]), ("RCDC", STYLE["c_green"])]):
    bars = ax.barh(y + (i - 0.5) * h, comp[src].values, height=h, label=src,
                   color=col, edgecolor=STYLE["edgecolor"], linewidth=0.5)
    for b, v in zip(bars, comp[src].values):
        if v:
            ax.text(v + 1, b.get_y() + b.get_height() / 2, str(int(v)),
                    va="center", fontsize=STYLE["annot_fs"] - 1)
ax.set_yticks(y); ax.set_yticklabels(comp.index)
ax.set_xlabel("Number of trials — MeSH vs RCDC over the same chapters",
              fontsize=STYLE["label_fs"])
set_title(ax, "b.", fontsize=STYLE["title_fs"], loc="left", fontweight="bold")
ax.legend(fontsize=STYLE["legend_fs"], frameon=False); ax.margins(x=0.12)
fig.tight_layout(w_pad=4); savefig(fig, "ct_rcdc_icd")
plt.show()

print(f"RCDC : {n_rcdc_icd}/{len(df_ct)} trials mapped to >=1 ICD-10 chapter "
      f"({len(rcdc_icd)} chapters touched)")
print(f"MeSH : {n_mesh_icd}/{len(df_ct)} trials mapped to >=1 ICD-10 chapter "
      f"({len(mesh_icd)} chapters touched)")

# audit — disease RCDC tags the rules leave unmapped (empty = full coverage)
unmapped = Counter()
for ts in df_ct["rcdc_disease"]:
    for t in set(ts):
        if map_icd_chapter_rules(t, ICD_CHAPTERS_RCDC) is None:
            unmapped[t] += 1
print(f"\nUnmapped disease RCDC tags: {len(unmapped)}")
if unmapped:
    display(pd.Series(unmapped, dtype="int64").sort_values(ascending=False)
              .head(20).rename("n_trials").to_frame())
display(comp.sort_values("RCDC", ascending=False))

### 3.5 MeSH, rebuilt from the registry — condition-**leaf** terms

The §3.1 ranking has a problem worth spelling out, because it changes what that figure means.

**`mesh_terms` is not indexer-assigned MeSH.** ClinicalTrials.gov *derives* it from the
trial's free-text `conditions` / `interventions` and returns it in **three separate buckets**,
which Dimensions flattens into one column:

| bucket | what it holds | example (NCT01315639, an Alzheimer's trial) |
|---|---|---|
| `conditionBrowseModule.meshes` | **leaf** terms actually matched — *the diseases* | Alzheimer Disease · Cognitive Dysfunction · Dementia |
| `conditionBrowseModule.ancestors` | MeSH-**tree scaffolding** | Pathologic Processes · Signs and Symptoms · Nervous System Diseases |
| `interventionBrowseModule.*` | terms from the **intervention** text | Magnetic Resonance Spectroscopy · Biomarkers |

Flattened together, the scaffolding wins: the most common "disease" in §3.1 is **Pathologic
Processes** (42 trials), followed by *Pathological Conditions, Signs and Symptoms* (35). Those
are tree ancestors, not findings.

So we **re-fetch the browse module** for every trial from the ClinicalTrials.gov REST API (v2,
no key needed) and keep the buckets apart. `src/utils/clinical_trials_fetch_ctgov.py` does the
fetch, caches the raw response to `data/non_academic/clinical_trials/ctgov_browse.pkl`, and
writes three columns back into the CSV — `mesh_leaf`, `mesh_ancestors`, `mesh_intervention` —
leaving the original `mesh_terms` untouched so the two can be compared. Re-run it after any
refresh of the trials file:

```bash
python src/utils/clinical_trials_fetch_ctgov.py    -- refresh     # cached; only new trials are fetched
```

**What the re-fetch settles about the missing 8%.** Of the 10 trials with no `mesh_terms`,
one is an **ISRCTN** record (no NLM browse module exists — structurally unfetchable) and three
come back populated, of which only **COPD** is a disease; the other two return *Health
Behavior* / *Motor Activity*. The remaining six stay empty because their `conditions` are
`Healthy`, `Aging`, `Health behavior`, `Health promotion`, `Body composition` — trials that
genuinely **study no disease**. That blank is a *finding*, not a gap: **imputing a disease onto
them (e.g. from RCDC) would fabricate one**, so we don't. Coverage is quoted on the trials that
have a disease, and the disease-free trials are reported as such.

In [ ]:
#!python src/utils/clinical_trials_fetch_ctgov.py    --refresh  

In [ ]:
df_ct["mesh_leaf"].isnull().sum()

In [ ]:
# =============================================================================
# LEAF MeSH — the disease terms NLM actually matched (see src/utils/clinical_trials_fetch_ctgov.py)
# =============================================================================
# The three columns below are written into the CSV by that module; if they are absent, run it:
#     python src/utils/clinical_trials_fetch_ctgov.py
assert "mesh_leaf" in df_ct.columns, "run: python src/utils/clinical_trials_fetch_ctgov.py"

# A handful of leaf terms are still not diseases — CTgov maps condition text like "physical
# activity" / "healthy" onto behavioural or descriptive MeSH. Same treatment as RCDC_STOP.
MESH_LEAF_STOP = {
    "Motor Activity", "Health Behavior", "Genetic Risk Score", "Psychological Well-Being",
    "Weight Loss", "Chronic Disease", "Body Weight", "Life Style", "Exercise",
}

df_ct["mesh_leaf_disease"] = df_ct["mesh_leaf"].apply(
    lambda c: [t for t in item_names(c) if t not in MESH_LEAF_STOP])

n_ship = int(df_ct["mesh_terms"].apply(lambda x: len(parse_listcol(x)) > 0).sum())
n_leaf = int(df_ct["mesh_leaf"].apply(lambda x: len(parse_listcol(x)) > 0).sum())
n_leafd = int(df_ct["mesh_leaf_disease"].str.len().gt(0).sum())
med_ship = df_ct["mesh_terms"].apply(lambda x: len(parse_listcol(x))).median()
med_leaf = df_ct["mesh_leaf"].apply(lambda x: len(parse_listcol(x))).median()
print(f"trials with MeSH   — shipped (leaf+ancestors+interventions): {n_ship}/{len(df_ct)}"
      f"  |  registry leaf: {n_leaf}/{len(df_ct)}  |  leaf & disease: {n_leafd}/{len(df_ct)}")
print(f"terms per trial    — shipped median {med_ship:.0f}  |  leaf median {med_leaf:.0f}"
      f"   (the difference is tree scaffolding + intervention terms)")
print(f"distinct terms     — shipped {count_items(df_ct['mesh_terms']).size}"
      f"  |  leaf {count_items(df_ct['mesh_leaf']).size}")

# ---- the clean disease bars: leaf terms, raw vs fractional -------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_hbar(count_items(df_ct["mesh_leaf_disease"], top=15), "a.",
          "Number of trials", color=STYLE["c_primary"], ax=axes[0],ylabel="MeSH leaf disease")
plot_hbar(count_items_frac(df_ct["mesh_leaf_disease"]).head(15),
          "b.", "Fractional trial weight",
          color=STYLE["c_gold"], fmt="{:.1f}", ax=axes[1])
fig.tight_layout(w_pad=3); savefig(fig, "ct_diseases_mesh_leaf")
plt.show()

# ---- what the shipped column was actually showing ---------------------------
print("The terms §3.1 ranked that are NOT diseases (tree ancestors / intervention terms):")
scaffold = count_items(df_ct["mesh_ancestors"]).head(6).rename("n_trials (ancestors)")
interv = count_items(df_ct["mesh_intervention"]).head(6).rename("n_trials (interventions)")
display(pd.concat([scaffold.reset_index(), interv.reset_index()], axis=1))

In [ ]:
# ancestors 

assert "mesh_leaf" in df_ct.columns, "run: python src/utils/clinical_trials_fetch_ctgov.py"

# A handful of leaf terms are still not diseases — CTgov maps condition text like "physical
# activity" / "healthy" onto behavioural or descriptive MeSH. Same treatment as RCDC_STOP.
MESH_LEAF_STOP = {
    "Motor Activity", "Health Behavior", "Genetic Risk Score", "Psychological Well-Being",
    "Weight Loss", "Chronic Disease", "Body Weight", "Life Style", "Exercise",
}

df_ct["mesh_ancestors_condition"] = df_ct["mesh_ancestors"].apply(
    lambda c: [t for t in item_names(c) if t not in MESH_LEAF_STOP])

n_ship = int(df_ct["mesh_terms"].apply(lambda x: len(parse_listcol(x)) > 0).sum())
n_leaf = int(df_ct["mesh_ancestors"].apply(lambda x: len(parse_listcol(x)) > 0).sum())
n_leafd = int(df_ct["mesh_ancestors_condition"].str.len().gt(0).sum())
med_ship = df_ct["mesh_terms"].apply(lambda x: len(parse_listcol(x))).median()
med_leaf = df_ct["mesh_ancestors"].apply(lambda x: len(parse_listcol(x))).median()
print(f"trials with MeSH   — shipped (leaf+ancestors+interventions): {n_ship}/{len(df_ct)}"
      f"  |  registry leaf: {n_leaf}/{len(df_ct)}  |  leaf & disease: {n_leafd}/{len(df_ct)}")
print(f"terms per trial    — shipped median {med_ship:.0f}  |  leaf median {med_leaf:.0f}"
      f"   (the difference is tree scaffolding + intervention terms)")
print(f"distinct terms     — shipped {count_items(df_ct['mesh_terms']).size}"
      f"  |  leaf {count_items(df_ct['mesh_ancestors']).size}")

# ---- the clean disease bars: leaf terms, raw vs fractional -------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_hbar(count_items(df_ct["mesh_ancestors_condition"], top=15), "a.",
          "Number of trials", color=STYLE["c_primary"], ax=axes[0],ylabel="MeSH Tree Ancestors")
plot_hbar(count_items_frac(df_ct["mesh_ancestors_condition"]).head(15),
          "b.", "Fractional trial weight",
          color=STYLE["c_gold"], fmt="{:.1f}", ax=axes[1])
fig.tight_layout(w_pad=3); savefig(fig, "ct_diseases_mesh_ancestors")
plt.show()


#### 3.5.2 MeSH leaf → ICD-10, via MeSH's own hierarchy (not keywords)

§3.3 mapped disease names to ICD chapters with a **keyword list** (`"heart"` → Circulatory).
That was always a guess, and it failed in three ways: substring accidents (`"alcohol"` fired
on *NON-alcoholic Fatty Liver Disease*, putting every NAFLD trial in Mental & behavioural);
chapter *order* silently deciding ties; and it only knew the words someone thought to type —
**74 of 220** leaf terms went unmapped, including real conditions like *Wounds and Injuries*
and *Premature Birth*.

For the MeSH leaf axis we don't have to guess any more. ClinicalTrials.gov returns each
condition's **MeSH descriptor id** (`Pulmonary Disease, Chronic Obstructive` = `D029424`), the
re-fetch now stores it (`mesh_leaf_ids`), and every descriptor carries **tree numbers**
(`C08.381.495.389`) whose top-level category maps cleanly onto an ICD chapter. `src/utils/mesh_tree.py`
pulls the trees from NLM's **public MeSH SPARQL endpoint** (no key, no UMLS licence), caches
them, and applies the map in `MESH_CAT_TO_ICD` — which is a table you can read and argue with.

| | keyword route | descriptor → tree route |
|---|---|---|
| trials mapped to ≥1 chapter | 150 / 195 | **169 / 195** |
| distinct terms/descriptors mapped | 146 / 220 (66%) | **209 / 227 (92%)** |
| *Wounds and Injuries* | unmapped | XIX Injury |
| *Premature Birth* | unmapped | XV Pregnancy |
| *Motor Activity*, *Smoking Cessation* | unmapped (by luck) | **unmapped by construction** (F01 = behaviour, not disease) |

Three design decisions, all visible in `mesh_tree.py`:

* **Behaviour trees (F01) map to nothing.** *Motor Activity*, *Sedentary Behavior* and
  *Smoking Cessation* are not diseases, so they get no chapter — the hierarchy enforces what
  the `MESH_LEAF_STOP` list previously had to do by hand.
* **The generic C23 branch (*Pathological Conditions, Signs and Symptoms*) is a last resort.**
  A descriptor is placed by its specific body-system tree if it has one; only descriptors that
  have *nothing but* C23 (*Inflammation*, *Frailty*, *Fibrosis*, *Genetic Predisposition*) land
  in chapter XVIII. That is why XVIII is suddenly visible — those trials existed before, they
  were simply invisible.
* **Supplementary Concept Records are followed, not dropped.** Rare-disease concepts
  (*Prostate cancer, familial*) carry no trees of their own; we resolve them through their
  `preferredMappedTo` heading (→ *Prostatic Neoplasms* → II Neoplasms).

The **keyword map is still used** — but only for the two axes that have no descriptor ids: the
shipped `mesh_terms` column (names only) and RCDC. Those remain approximations, and the three-axis
comparison below now mixes one principled mapping with two keyword ones, which is worth
remembering when reading it.

> Re-running the pipeline: `python src/utils/clinical_trials_fetch_ctgov.py --refresh` (stores
> `mesh_leaf_ids`), then this cell fetches and caches the MeSH trees on first run.

In [ ]:
# =============================================================================
# LEAF MeSH -> ICD-10 chapters — via MeSH's OWN HIERARCHY, not keywords
# =============================================================================
# The keyword map (§3.3/§3.4.1) guesses a chapter from the term STRING. For the MeSH leaf
# axis we no longer have to guess: CTgov gives us each condition's MeSH descriptor id
# (D029424 = COPD), and every descriptor carries tree numbers (C08.381.495.389) whose
# top-level category maps onto an ICD chapter. src/utils/mesh_tree.py fetches the trees from
# NLM's public SPARQL endpoint (cached to disk) and applies that map.
#
# Why it is better than the keyword list:
#   * no substring accidents (it was 'alcohol' in 'NON-alcoholic Fatty Liver Disease' that
#     put every NAFLD trial in Mental & behavioural);
#   * it knows terms nobody thought to type — Wounds and Injuries -> XIX, Premature Birth ->
#     XV, Memory Disorders -> VI;
#   * behaviour-only descriptors (F01: Motor Activity, Smoking Cessation, Sedentary Behavior)
#     map to NOTHING, which is correct — they are not diseases;
#   * supplementary concepts (rare diseases, no trees of their own) are resolved through
#     their `preferredMappedTo` heading instead of being dropped.
#
# The KEYWORD map is kept for the two axes that have no descriptor ids: the shipped
# `mesh_terms` column (names only) and RCDC.
from utils.data_analysis_04_non_academic_clinical_trials_mesh_tree import fetch_tree_numbers, descriptor_chapters, audit

if "mesh_leaf_ids" not in df_ct.columns:
    raise FileNotFoundError(
        "Missing clinical-trial MeSH descriptor IDs (mesh_leaf_ids). Restore the enriched "
        "clinical_trials.csv and mesh_tree_numbers.pkl; the Showcase+ snapshot contains "
        "term names but cannot recover descriptor IDs. "
        "See src/utils/data_creation_clinical_trials_fetch_ctgov.py for the separate enrichment step."
    )

# IMPORTANT: filter the descriptor ids by the SAME stop-list applied to the names
# (mesh_leaf_disease). Otherwise non-disease leaves (Genetic Risk Score, Body Weight, ...)
# leak into the ICD chapters through the id path — they all carry only a C23 tree, so they
# spuriously inflate chapter XVIII (Symptoms & signs). Names and ids are position-aligned.
def _leaf_ids_disease(row):
    return [d for n, d in zip(item_names(row["mesh_leaf"]), parse_listcol(row["mesh_leaf_ids"]))
            if n not in MESH_LEAF_STOP]
df_ct["mesh_leaf_id_list"] = df_ct.apply(_leaf_ids_disease, axis=1)
MESH_TREE = fetch_tree_numbers(
    {d for L in df_ct["mesh_leaf_ids"].apply(parse_listcol) for d in L},
    allow_network=False,
)

# per-trial ICD chapters from the descriptors (the set a trial touches)
df_ct["icd_leaf"] = df_ct["mesh_leaf_id_list"].apply(
    lambda L: sorted(descriptor_chapters(L, MESH_TREE)))

leaf_icd = (pd.Series(Counter(ch for S in df_ct["icd_leaf"] for ch in S), dtype="int64")
              .sort_index().sort_values(ascending=False, kind="stable"))
n_leaf_icd = int(df_ct["icd_leaf"].apply(len).gt(0).sum())

# the two axes WITHOUT descriptor ids still use the keyword rules
ship_icd, n_ship_icd = icd_chapter_counts(df_ct["mesh_terms"])
rcdc_icd2, n_rcdc_icd2 = icd_chapter_counts(df_ct["rcdc_disease"])

# what the old keyword route made of the same leaf terms — the honest before/after
_kw_leaf, _n_kw_leaf = icd_chapter_counts(df_ct["mesh_leaf_disease"])
print(f"MeSH leaf -> ICD : descriptor/tree route maps {n_leaf_icd}/{len(df_ct)} trials; "
      f"the keyword route managed {_n_kw_leaf}/{len(df_ct)}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_hbar(leaf_icd, "a.", "Number of trials — by ICD-10 chapter (MeSH leaf, descriptor route)",
          color=STYLE["c_primary"], pct_of=len(df_ct), ax=axes[0])

# (b) the three axes over the same chapters
comp3 = (pd.concat([ship_icd.rename("MeSH (shipped)"), leaf_icd.rename("MeSH (leaf)"),
                    rcdc_icd2.rename("RCDC")], axis=1)
           .fillna(0).astype(int).sort_values("MeSH (leaf)"))
ax = axes[1]
y, h = np.arange(len(comp3)), 0.26
for i, (src, col) in enumerate([("MeSH (shipped)", STYLE["colors"][3]),
                                ("MeSH (leaf)", STYLE["c_primary"]),
                                ("RCDC", STYLE["c_green"])]):
    bars = ax.barh(y + (i - 1) * h, comp3[src].values, height=h, label=src,
                   color=col, edgecolor=STYLE["edgecolor"], linewidth=0.5)
    for b, v in zip(bars, comp3[src].values):
        if v:
            ax.text(v + 0.8, b.get_y() + b.get_height() / 2, str(int(v)),
                    va="center", fontsize=STYLE["annot_fs"] - 2)
ax.set_yticks(y); ax.set_yticklabels(comp3.index)
# NOT "identical rules" any more: the leaf axis is mapped by MeSH's hierarchy, the other
# two by the keyword list (they have no descriptor ids). Say so on the axis.
ax.set_xlabel("Number of trials — three disease axes "
              "(leaf: MeSH tree · shipped & RCDC: keyword)",
              fontsize=STYLE["label_fs"])
set_title(ax, "b.",
             fontsize=STYLE["title_fs"], loc="left", fontweight="bold")
ax.legend(fontsize=STYLE["legend_fs"] - 1, frameon=False, loc="lower right")
ax.margins(x=0.14)
fig.tight_layout(w_pad=4); savefig(fig, "ct_icd_three_axes")
plt.show()

for lab, n, how in [("MeSH (shipped)", n_ship_icd, "keyword"),
                    ("MeSH (leaf)", n_leaf_icd, "MeSH tree"),
                    ("RCDC", n_rcdc_icd2, "keyword")]:
    print(f"{lab:<15}: {n}/{len(df_ct)} trials mapped to >=1 ICD-10 chapter   [{how}]")

# audit — every descriptor, its trees, and the chapter(s) they resolve to
_aud = audit({d for L in df_ct["mesh_leaf_id_list"] for d in L}, MESH_TREE)
_unmapped = _aud[_aud["chapters"] == "— none —"]
print(f"\nDescriptors: {len(_aud)} | mapped {len(_aud) - len(_unmapped)} | "
      f"not a disease / unmapped {len(_unmapped)} (behaviour & generic-symptom trees)")
print("\nThe shipped column maps MORE trials only because its ancestor terms "
      "(e.g. 'Nervous System Diseases') match a chapter even when no disease was studied.")
display(comp3.sort_values("MeSH (leaf)", ascending=False))

In [ ]:
169/195

#### 3.5.1 The trials with **no** `mesh_terms` — what are they actually about?

Dimensions ships **no MeSH at all** for a handful of trials. Rather than treat that as a hole
to be filled, the cell below reads what those trials *say* (`conditions`, `brief_title`,
`registry`, `start_date`, RCDC) and sorts them into **why** the MeSH is absent. The reasons
are derived from the data, not hand-typed per trial, so this survives a refresh of the
trials file:

| reason | rule | what it means |
|---|---|---|
| **non-CTgov registry** | `registry != ClinicalTrials.gov` | ISRCTN/EU-CTR records never get an NLM browse module — MeSH is *structurally* unavailable, not missing |
| **recovered by re-fetch** | registry leaf MeSH now non-empty | the Dimensions snapshot pre-dates NLM's indexing; §3.5 already pulled the terms back |
| **too recent to be indexed** | `start_year >= 2025` | registrations in the final analysis year (future starts are excluded). NLM has not generated the browse module yet — these will populate on their own |
| **studies no disease** | everything else | `conditions` is `Healthy`, `Aging`, `Health promotion`, `Body composition`, `Post menopause`… — healthy-volunteer physiology, behaviour and life-stage trials |

That last group is the point worth carrying into the write-up: **the blank is a finding, not a
gap.** Imputing a disease onto a healthy-volunteer salt-intake trial or a twins MRI ageing
study would manufacture a disease focus that the trial does not have.

In [ ]:
# =============================================================================
# WHY is mesh_terms missing? — classify, don't impute
# =============================================================================
def _no_mesh_reason(r):
    """Data-derived reason a trial carries no Dimensions MeSH (rules, not a hard-coded list)."""
    if str(r["registry"]).strip() != "ClinicalTrials.gov":
        return "1. non-CTgov registry (no NLM MeSH exists)"
    if len(parse_listcol(r["mesh_leaf"])) > 0:
        return "2. recovered by the §3.5 re-fetch"
    yr = r["start_year"]
    if pd.notna(yr) and int(yr) >= 2025:
        return "3. too recent — not yet indexed by NLM"
    return "4. studies no disease (healthy / physiology / behaviour)"


no_mesh = df_ct[df_ct["mesh_terms"].isna()].copy()
no_mesh["reason"] = no_mesh.apply(_no_mesh_reason, axis=1)
no_mesh["conditions_txt"] = no_mesh["conditions"].apply(lambda x: "; ".join(parse_listcol(x)))
no_mesh["rcdc_top"] = no_mesh["rcdc_disease"].apply(lambda ts: ", ".join(ts[:4]) or "—")
no_mesh["leaf_now"] = no_mesh["mesh_leaf"].apply(lambda x: ", ".join(item_names(x)) or "—")

print(f"Trials with NO Dimensions mesh_terms: {len(no_mesh)}/{len(df_ct)}\n")
print(no_mesh["reason"].value_counts().sort_index().to_string())
print("\nEvery one of them, and what it is about:")
with pd.option_context("display.max_colwidth", 60, "display.width", 250):
    display(no_mesh.sort_values("reason")[
        ["id", "reason", "brief_title", "conditions_txt", "leaf_now", "rcdc_top",
         "start_year", "study_type"]
    ].rename(columns={"conditions_txt": "conditions (free text)",
                      "leaf_now": "registry MeSH leaf (now)",
                      "rcdc_top": "RCDC disease tags"}).reset_index(drop=True))

print("\nEvery one of these trials still carries RCDC disease tags "
      f"({int(no_mesh['rcdc_disease'].str.len().gt(0).sum())}/{len(no_mesh)}), which is why "
      "RCDC (§3.4) covers more trials than any MeSH route — but a disease-free trial with a\n"
      "generic RCDC tag is still a disease-free trial. We report the blank; we do not fill it.")

### 3.6 What each disease is *called*, and where it sits in ICD — two heatmaps

Both panels share the **same y-axis**: the top-10 **MeSH leaf diseases** (§3.5), ranked by trial count, with each row's trial count in the label.

* **(a) disease × free-text `conditions`.** The rows are the *controlled* vocabulary NLM
  derived; the columns are what the sponsor actually **typed** into the registry, normalised
  with the same `norm_key` + synonym map as the §3.2 word cloud. The cell is the **% of that
  disease's trials whose `conditions` carry that phrase**. The strong cells on the "diagonal"
  are expected — `mesh_leaf` is *derived from* `conditions`, so this is partly a picture of
  the mapping itself (which surface forms NLM folded into which MeSH descriptor). The
  informative part is **off-diagonal**: the *other* things a disease's trials are also about
  — e.g. what obesity trials co-list alongside obesity.
* **(b) disease × ICD-10 chapter.** The same rows against the ICD chapters, using the §3.4.1
  rules. The diagonal is 100% by construction (an *Atrial Fibrillation* trial is always
  Circulatory); the informative part is again **off-diagonal** — the co-morbidity spread of
  each disease's trials.

Both use `Spectral_r` (cool = rare → warm = common) and are drawn as one figure.

> Percentages are over the trials **in that row**, and the tail rows are small (`n` is in each
> label) — read the bottom rows as indicative, not as rates.

In [ ]:
# =============================================================================
# §3.6  MeSH leaf x {free-text conditions, ICD chapters} — one figure, shared y-axis
# =============================================================================
import textwrap

# ---- per-trial NORMALISED conditions -----------------------------------------
# Same normalisation as the §3.2 word cloud (norm_key + COND_SYNONYMS), so the labels here
# and there are the same objects. normalized_condition_freq() only returns totals, so we
# rebuild the key -> display-label map (canonical synonym if any, else the commonest
# surface form) and then tag each trial with its labels.
def build_cond_label_map(series):
    """merge-key -> display label, exactly as normalized_condition_freq labels them."""
    surf = defaultdict(Counter)
    for cell in series:
        for it in item_names(cell):
            surf[norm_key(it)][html.unescape(it)] += 1
    return {k: (k if k in _CANON else c.most_common(1)[0][0]) for k, c in surf.items()}


COND_LABEL = build_cond_label_map(df_ct["conditions"])
df_ct["cond_norm"] = df_ct["conditions"].apply(
    lambda cell: sorted({COND_LABEL[norm_key(it)] for it in item_names(cell)}))

TOP_LEAF, TOP_COND = 10, 10

# rows = the TOP_LEAF diseases by RAW trial count (so the row order matches the n= in each
# label; fractional weight would push e.g. COVID-19 above higher-count Cognitive Dysfunction).
leaf_rank = count_items(df_ct["mesh_leaf_disease"]).head(TOP_LEAF)
leaves = list(leaf_rank.index)
leaf_trials = {L: df_ct[df_ct["mesh_leaf_disease"].apply(lambda T: L in T)] for L in leaves}

# (a) columns: the commonest normalised conditions among the disease-coded trials
# raw trial count (consistent with the row ranking), restricted to disease-coded trials
conditions = list(count_items(
    df_ct.loc[df_ct["mesh_leaf_disease"].apply(len).gt(0), "cond_norm"]
).head(TOP_COND).index)

# (b) columns: the ICD chapters these diseases actually touch, most-used first.
# `icd_leaf` comes from §3.5 — the MeSH descriptor -> tree -> chapter route, not keywords.
_ch_count = Counter()
for sub in leaf_trials.values():
    for S in sub["icd_leaf"]:
        _ch_count.update(sorted(S))
chapters = [c for c, _ in sorted(_ch_count.items(), key=lambda item: (-item[1], item[0]))]
chapters = chapters[0:TOP_COND] if len(chapters) > TOP_COND else chapters
H_cond = pd.DataFrame(
    [[100 * sub["cond_norm"].apply(lambda C: c in C).mean() for c in conditions]
     for sub in leaf_trials.values()], index=leaves, columns=conditions)
H_icd = pd.DataFrame(
    [[100 * sub["icd_leaf"].apply(lambda S: ch in S).mean() for ch in chapters]
     for sub in leaf_trials.values()], index=leaves, columns=chapters)

print(f"Rows: top {len(leaves)} MeSH leaf diseases  |  "
      f"(a) cols: top {len(conditions)} normalised free-text conditions  |  "
      f"(b) cols: {len(chapters)} ICD-10 chapters touched")

# ---- draw --------------------------------------------------------------------
n = len(leaves)
# n on the same line, wrapped as one string: the two-line form collided row-to-row
ylabels = [textwrap.fill(f"{L} \n", 25)+f"\n(n={len(leaf_trials[L])})" for L in leaves][::-1]
# ylabels = [textwrap.fill(f"{L} \n(n={len(leaf_trials[L])})", 32) for L in leaves][::-1]
short_chap = [c.split("(")[0].strip() for c in chapters]     # 'IX  Circulatory system'

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 5), sharey=True, layout="constrained",
                               gridspec_kw={"width_ratios": [1, 1]})

# Both panels are the SAME quantity on the SAME 0-100 scale (% of the row's trials), so they
# share one colour scale and ONE colourbar — two bars would imply two different scales.
for i, ax, M, cols, title in [
        (0, ax0, H_icd.values[::-1], short_chap,
         "a."),
        (1, ax1, H_cond.values[::-1], conditions,
         "b.")]:
    im = ax.imshow(M, aspect="auto", cmap="Spectral_r", origin="lower", vmin=0, vmax=100,
                   extent=(-0.5, len(cols) - 0.5, -0.5, n - 0.5))
    if i==0:
        ax.set_xticks(range(len(cols)))
        ax.set_ylabel("MeSH leaf disease", fontsize=STYLE["label_fs"])
        #ax.set_xlabel("Normalised free-text condition", fontsize=STYLE["label_fs"])
        ax.set_xlabel("ICD-10 chapter", fontsize=STYLE["label_fs"])
    else:
        ax.set_xticks(range(len(cols)))
        
        ax.text(0.5, -0.5, "Normalised free-text condition", fontsize=STYLE["label_fs"],
                ha="center", va="top", transform=ax.transAxes)
   
    ax.set_xticklabels([textwrap.fill(c, 20) for c in cols], rotation=90, ha="center",
                       fontsize=STYLE["tick_fs"] - 2)
    for i in range(n):
        for j in range(len(cols)):
            ax.text(j, i, f"{M[i, j]:.1f}" if M[i, j]>0 else '-', ha="center", va="center", #  if M[i, j]>0 else ''
                    fontsize=STYLE["annot_fs"] - 3,
                    color="white" if M[i, j] > 72 else "black")
    # add grid
    set_title(ax, title, loc="left", fontsize=12, fontweight="bold")

# one horizontal colourbar under both panels
cb = fig.colorbar(im, ax=[ax0, ax1], #orientation="horizontal", location="bottom",
                  fraction=0.05, pad=0.02, aspect=40)
cb.set_label("% of the disease's trials touching \nthe chapter / condition",
             fontsize=STYLE["label_fs"] - 1)

ax0.set_yticks(np.arange(n))
ax0.set_yticklabels(ylabels, fontsize=STYLE["tick_fs"] - 3)

savefig(fig, "ct_mesh_leaf_condition_icd_heatmaps")
plt.show()

## 4. Status — aggregated lifecycle stage

We report status only at the **coarse lifecycle stage** (not the raw 9 registry labels),
split by interventional vs. observational `study_type`. The 9→5 mapping:

| stage | registry `overall_status` values folded in |
|---|---|
| **Planned** | Not yet recruiting |
| **Ongoing** | Recruiting · Enrolling by invitation · Active, not recruiting |
| **Completed** | Completed |
| **Stopped early** | Terminated · Withdrawn · Suspended |
| **Unknown** | Unknown status |

Each bar is annotated with its count; **Σ** is the per-stage total.

In [ ]:
def plot_grouped_status(df, status_col, group_col, style=STYLE, ax=None,
                        title="", order=None, name="status"):
    """Horizontal grouped bars with per-bar counts and a bold per-status Σ total."""
    tab = df.groupby([status_col, group_col]).size().unstack(fill_value=0)
    if order is not None:
        tab = tab.reindex([o for o in order if o in tab.index]).iloc[::-1]
    else:
        tab = tab.loc[tab.sum(axis=1).sort_values().index]
    groups = list(tab.columns)
    totals = tab.sum(axis=1)
    xmax = max(int(totals.max()), 1)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 0.7 * len(tab) + 1.6))
    else:
        fig = ax.figure
    y = np.arange(len(tab)); h = 0.8 / max(len(groups), 1)
    for i, g in enumerate(groups):
        bars = ax.barh(y + i*h - 0.4 + h/2, tab[g].values, height=h,
                       label=str(g), color=style["colors"][i % len(style["colors"])],
                       edgecolor=style["edgecolor"], linewidth=0.5)
        for b, v in zip(bars, tab[g].values):
            if v > 0:
                ax.text(v + xmax*0.01, b.get_y() + b.get_height()/2, f"{int(v)}",
                        va="center", fontsize=style["annot_fs"] - 1)
    for yi, tot in zip(y, totals.values):
        ax.text(xmax*1.08, yi, f"Total {int(tot)}", va="center", ha="left",
                fontsize=style["annot_fs"], fontweight="bold")
    ax.set_yticks(y); ax.set_yticklabels(tab.index)
    ax.set_xlim(0, xmax * 1.22)
    ax.set_xlabel("Number of trials", fontsize=style["label_fs"])
    set_title(ax, title, fontsize=style["title_fs"], fontweight="bold")
    ax.legend(title=group_col.replace("_", " ").title(), frameon=True,
              fontsize=style["legend_fs"],loc="lower right", bbox_to_anchor=(0.8, 0.02))
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax, tab


STATUS_STAGE = {
    "Not yet recruiting":      "Planned",
    "Recruiting":              "Ongoing",
    "Enrolling by invitation": "Ongoing",
    "Active, not recruiting":  "Ongoing",
    "Completed":               "Completed",
    "Terminated":              "Stopped early",
    "Withdrawn":               "Stopped early",
    "Suspended":               "Stopped early",
    "Unknown status":          "Unknown",
}
STAGE_ORDER = ["Planned", "Ongoing", "Completed", "Stopped early", "Unknown"]
df_ct["status_stage"] = df_ct["overall_status"].map(STATUS_STAGE).fillna("Unknown")

_, _ax, stage_tab = plot_grouped_status(
    df_ct, "status_stage", "study_type", order=STAGE_ORDER,
    title="", name="ct_status_stage")
_ax.set_xlabel("Number of trials — lifecycle stage, by study type",
               fontsize=STYLE["label_fs"])
plt.show()


stage_tab.assign(Total=lambda t: t.sum(axis=1))

In [ ]:
def plot_grouped_status(df, status_col, group_col, style=STYLE, ax=None,
                        title="", order=None, name="status", show_legend=True):
    """Horizontal grouped bars with per-bar counts and a bold per-status Σ total."""
    tab = df.groupby([status_col, group_col]).size().unstack(fill_value=0)

    if order is not None:
        tab = tab.reindex([o for o in order if o in tab.index]).iloc[::-1]
    else:
        tab = tab.loc[tab.sum(axis=1).sort_values().index]

    groups = list(tab.columns)
    totals = tab.sum(axis=1)

    # max single bar length, used for annotation spacing
    bar_max = max(int(tab.to_numpy().max()), 1)

    # longest bar in each row (use this to place Σ just to the right)
    row_max = tab.max(axis=1)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 0.7 * len(tab) + 1.6))
    else:
        fig = ax.figure

    y = np.arange(len(tab))
    h = 0.8 / max(len(groups), 1)

    for i, g in enumerate(groups):
        bars = ax.barh(
            y + i * h - 0.4 + h / 2,
            tab[g].values,
            height=h,
            label=str(g),
            color=style["colors"][i % len(style["colors"])],
            edgecolor=style["edgecolor"],
            linewidth=0.5,
            
        )

        for b, v in zip(bars, tab[g].values):
            if v > 0:
                ax.text(
                    v + bar_max * 0.01,
                    b.get_y() + b.get_height() / 2,
                    f"{int(v)}",
                    va="center",
                    fontsize=style["annot_fs"] - 1,
                )

    # place Σ next to the longest bar in each row
    row_max = tab.max(axis=1)

    offset = 4
    bracket_h = h * (len(groups) - 0.2)   # span almost the whole grouped bar height
    bracket_w = 0.4                        # horizontal tick length

    for yi, tot, rmax in zip(y, totals.values, row_max.values):

        x = rmax + offset

        # vertical line
        ax.plot(
            [x, x],
            [yi - bracket_h/2, yi + bracket_h/2],
            color="black",
            lw=0.6,
            clip_on=False,
        )
        
        # vertical line
        ax.plot(
            [x , x+bracket_w],
            [yi,yi],
            color="black",
            lw=0.6,
            clip_on=False,
        )

        # top tick
        ax.plot(
            [x - bracket_w, x],
            [yi - bracket_h/2, yi - bracket_h/2],
            color="black",
            lw=0.6,
            clip_on=False,
        )

        # bottom tick
        ax.plot(
            [x - bracket_w, x],
            [yi + bracket_h/2, yi + bracket_h/2],
            color="black",
            lw=0.6,
            clip_on=False,
        )

        # Σ label
        ax.text(
            x + 0.6,
            yi,
            f"Total {int(tot)}",
            va="center",
            ha="left",
            fontsize=style["annot_fs"],
            fontweight="bold",
        )
    ax.set_yticks(y)
    ax.set_yticklabels(tab.index)
    ax.set_xlabel("Number of trials", fontsize=style["label_fs"])

    # leave enough room on the right for Σ labels and legend
    xmax = max(int((row_max + offset).max()), bar_max)
    ax.set_xlim(0, xmax * 1.20)

    if show_legend:
        ax.legend(
            title=group_col.replace("_", " ").title(),
            frameon=True,
            fontsize=style["legend_fs"],
            loc="lower right",
            edgecolor="black",
        )

    if own_fig:
        fig.tight_layout()
        savefig(fig, name, style)

    return fig, ax, tab


def plot_bar_over_time(counts_by_year, title, ylabel, style=STYLE, color=None,
                       ax=None, name="over_time"):
    color = color or style["c_green"]
    own_fig = ax is None

    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 5))
    else:
        fig = ax.figure

    ax.bar(
        counts_by_year.index.astype(int),
        counts_by_year.values,
        color=color,
        edgecolor=style["edgecolor"],
        linewidth=0.6,
    )

    for x, v in zip(counts_by_year.index.astype(int), counts_by_year.values):
        if v > 0:
            ax.text(
                x, v, f"{int(v)}",
                ha="center", va="bottom",
                fontsize=style["annot_fs"] - 3
            )

    set_title(ax, title, fontsize=style["title_fs"], fontweight="bold")
    ax.set_xlabel("Year", fontsize=style["label_fs"])
    ax.set_ylabel(ylabel, fontsize=style["label_fs"])

    if own_fig:
        fig.tight_layout()
        savefig(fig, name, style)

    return fig, ax


# ---- data prep ----
STATUS_STAGE = {
    "Not yet recruiting":      "Planned",
    "Recruiting":              "Ongoing",
    "Enrolling by invitation": "Ongoing",
    "Active, not recruiting":  "Ongoing",
    "Completed":               "Completed",
    "Terminated":              "Stopped early",
    "Withdrawn":               "Stopped early",
    "Suspended":               "Stopped early",
    "Unknown status":          "Unknown",
}
STAGE_ORDER = ["Planned", "Ongoing", "Completed", "Stopped early", "Unknown"]
df_ct["status_stage"] = df_ct["overall_status"].map(STATUS_STAGE).fillna("Unknown")

start_counts = (
    df_ct.dropna(subset=["start_year"])
         .astype({"start_year": int})
         .groupby("start_year")
         .size()
)
start_counts = start_counts.reindex(
    range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1),
    fill_value=0
)

# ---- one merged figure ----
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(10, 5),
    gridspec_kw={"width_ratios": [1, 1]},
    constrained_layout=True
)

plot_grouped_status(
    df_ct,
    "status_stage",
    "study_type",
    order=STAGE_ORDER,
    title="",
    ax=ax1,
    show_legend=True,
)

plot_bar_over_time(
    start_counts,
    "",
    "Number of trials started",
    color=STYLE["c_green"],
    ax=ax2,
)

# panel labels
ax1.text(0, 1.05, "A.", transform=ax1.transAxes, fontsize=STYLE['title_fs'], fontweight="bold", ha="left")
ax2.text(0, 1.05, "B.", transform=ax2.transAxes, fontsize=STYLE['title_fs'], fontweight="bold", ha="left")

# the panel titles now live on the x axes
ax1.set_xlabel("Number of trials", fontsize=STYLE["label_fs"])
ax1.set_ylabel("Lifecycle stage", fontsize=STYLE["label_fs"])
ax2.set_ylabel("Number of trials started", fontsize=STYLE["label_fs"])
ax2.set_xlabel("Trials start year", fontsize=STYLE["label_fs"])

# optional: save the combined figure once
savefig(fig, "ct_combined_figure", STYLE)
plt.show()

## 5. Recruitment size

How big are these trials? `study_participants` is the **planned enrollment** each trial
registered. It is heavily right-skewed, so we show it on a **log** axis with the median /
quartiles marked, plus a bucketed breakdown. (A handful of very large observational cohorts
dominate the raw total.)

In [ ]:
print(df_ct['study_participants'].sum())

print(df_ct['study_participants'].describe())

In [ ]:
df_ct.sort_values(by='study_participants',ascending=False).head(10)

## 7. Who / where — organisations by geography and sector

Where are the trials that cite UKB research run, and **who** runs them? **Two figures**:

* **a world map** — trials per country (continuous shading, no bins), with the **top-5
  countries outlined and labelled** (leader lines), in the authorship notebook's
  `plot_country_map` style; ISO-3 matched via `patent_utils`;
* **a sector breakdown** — each research org mapped to a **sector** (Academia / Healthcare /
  Industry / Government / Nonprofit / Other) via its Dimensions `types`, stacked per country.

> Title note: these organisations run **UKB-citing** trials — **UK Biobank is not the trial
> sponsor**, so the figure says "where UKB-citing trials are run", not "UKB trials".

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def to_iso3(code, name):
    """ISO-3 from an ISO-2 country_code when clean, else fuzzy-match the country name
    (reuses patent_utils, same matcher as the authorship / patents maps)."""
    iso2 = None
    if isinstance(code, str) and len(code.strip()) == 2:
        iso2 = code.strip().upper()
    if iso2 is None:
        iso2 = patent.to_iso2(name)
    if iso2 is None:
        return None
    return patent.iso2_to_iso3(iso2)


def ct_country_iso_counts(df):
    """Trials per country as an ISO-3-indexed Series (each country once per trial)."""
    c = Counter()
    for cell in df["research_orgs"]:
        iso = set()
        for o in parse_listcol(cell):
            if isinstance(o, dict) and o.get("country_name"):
                i = to_iso3(o.get("country_code"), o.get("country_name"))
                if i:
                    iso.add(i)
        c.update(iso)
    return pd.Series(c, dtype="int64").sort_values(ascending=False)


def plot_ct_country_map(counts_iso, title, style=STYLE, ax=None, label="Trials",
                        annotate_top=5, name="ct_country_map"):
    """Continuous-shaded choropleth (no bins) of a per-country trial count. Countries with
    no trials are light grey; a colourbar carries the scale. `annotate_top` highlights and
    labels the N highest countries (black outline + name + value + leader line), mirroring
    the authorship notebook's `plot_country_map` style."""
    from matplotlib.colors import LinearSegmentedColormap
    import geopandas as gpd
    world = gpd.read_file(P.WORLD_SHP)
    world.columns = [c.lower() for c in world.columns]
    world = world[world["admin"] != "Antarctica"]
    world["iso_key"] = world["iso_a3"].where(world["iso_a3"] != "-99", world["adm0_a3"])
    cdf = counts_iso.rename_axis("iso3").reset_index(name="value")
    merged = world.merge(cdf, how="left", left_on="iso_key", right_on="iso3")

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(10, 4))
    else:
        fig = ax.figure
    cmap = LinearSegmentedColormap.from_list("brand_seq", ["#EAF0F6", style["c_primary"]])
    merged.plot(color="#FFFFFF", ax=ax, edgecolor=style["edgecolor"], linewidth=0.3)  # base
    merged.dropna(subset=["value"]).plot(
        column="value", cmap=cmap, ax=ax, edgecolor=style["edgecolor"], linewidth=0.3,
        legend=True, legend_kwds={"label": label, "shrink": 0.5, "pad": 0.01})

    # highlight + label the top-N countries (black outline, leader lines, greedy stagger)
    if annotate_top:
        top = merged.dropna(subset=["value"]).nlargest(annotate_top, "value")
        top.boundary.plot(ax=ax, edgecolor="black", linewidth=1.4)
        placed = []
        def free_slot(x, y):
            # Wider search + wider "too close" box than a single map needs: in a 1x2 panel
            # the European top-5 (UK / Sweden / Netherlands) sit within a few degrees and
            # their label boxes overlapped at the tighter thresholds.
            for dy in (12, -12, 26, -26, 40, -40, 54, -54):
                for dx in (0, 22, -22, 42, -42, 62, -62):
                    cx, cy = x + dx, y + dy
                    if all(abs(cx - px) > 30 or abs(cy - py) > 14 for px, py in placed):
                        return cx, cy
            return x, y + 12
        for _, row in top.sort_values("value", ascending=False).iterrows():
            pt = row.geometry.representative_point()
            nm = row.get("name") or row.get("admin") or row["iso3"]
            lx, ly = free_slot(pt.x, pt.y); placed.append((lx, ly))
            ax.annotate(f"{nm}\n{int(row['value'])}", xy=(pt.x, pt.y), xytext=(lx, ly),
                        textcoords="data", ha="center", va="center",
                        fontsize=style["annot_fs"] - 1, fontweight="bold", color="black",
                        bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="black",
                                  lw=1.0, alpha=0.92),
                        arrowprops=dict(arrowstyle="-", lw=0.7, color="black"), zorder=6)
    ax.set_ylim(-58, 90)
    set_title(ax, title, fontsize=style["title_fs"]-3, fontweight="bold", loc="left")
    ax.axis("off")
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax


def country_sector_table(df, top_countries=10):
    """Fractional (country x sector) trial weights from research_orgs. Each trial spreads
    weight 1 across its unique (country, sector) org pairs -> columns sum to #located trials."""
    rows = []
    for cell in df["research_orgs"]:
        pairs = {(o.get("country_name"), org_sector(o))
                 for o in parse_listcol(cell)
                 if isinstance(o, dict) and o.get("country_name")}
        if not pairs:
            continue
        f = 1.0 / len(pairs)
        for c, s in pairs:
            rows.append((c, s, f))
    t = pd.DataFrame(rows, columns=["country", "sector", "w"])
    mat = t.pivot_table(index="country", columns="sector", values="w",
                        aggfunc="sum", fill_value=0)
    mat = mat.loc[mat.sum(axis=1).sort_values(ascending=False).index].head(top_countries)
    cols = [s for s in SECTOR_ORDER if s in mat.columns]
    return mat[cols]


def plot_stacked_sectors(mat, title, xlabel, style=STYLE, ax=None,
                         color_map=SECTOR_COLORS, name="ct_where_sector"):
    mat = mat.iloc[::-1]                              # largest on top (barh)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(10,7))
    else:
        fig = ax.figure
    y = np.arange(len(mat)); left = np.zeros(len(mat))
    for s in mat.columns:
        vals = mat[s].values
        ax.barh(y, vals, left=left, label=s, color=color_map.get(s, "#999"),
                edgecolor=style["edgecolor"], linewidth=0.4)
        left = left + vals
    totals = mat.sum(axis=1).values
    for yi, tot in zip(y, totals):
        ax.text(tot + totals.max()*0.01, yi, f"{tot:.0f}", va="center",
                fontsize=style["annot_fs"])
    ax.set_yticks(y); ax.set_yticklabels(mat.index)
    ax.set_xlabel(xlabel, fontsize=style["label_fs"])
    set_title(ax, title, fontsize=style["title_fs"], fontweight="bold")
    ax.legend(title="Sector", frameon=True, fontsize=style["legend_fs"], ncol=2,edgecolor="black")
    ax.margins(x=0.10)
    if own_fig:
        fig.tight_layout(); savefig(fig, name, style)
    return fig, ax


# Two separate figures (kept apart so the map keeps its natural aspect ratio).
# (1) annotated world map
iso_counts = ct_country_iso_counts(df_ct)
plot_ct_country_map(iso_counts, " ",#"Where UKB-citing trials are run (trials per country)",
                    label="Number of trials", annotate_top=5, name="ct_country_map")
plt.show()


In [ ]:
iso_counts

#### 7.1 Two geographies: where the **trials** run vs. where the **research** came from

The map above locates the *trials*. This pair sets it beside the geography of the **UK Biobank
papers those trials cite** — using the papers' **`research_orgs`** (the institutions credited on
the paper), *not* author affiliations, matched to ISO-3 with the same `to_iso3` used for the
trials, so the two panels are strictly comparable.

* **(a) Trials** — where the UKB-citing trials are run (each country counted once per trial).
* **(b) Cited UKB papers** — where the research those trials lean on was produced (each country
  once per paper).

Read together they separate **who produces the evidence** from **who acts on it**: a country
heavy in (b) but thin in (a) is exporting influence — its UKB findings are changing trial design
elsewhere.

In [ ]:
# =============================================================================
# §7.1  trials by country  vs  cited papers by country (research ORGS, not authors)
# =============================================================================
# §7 runs before §8 builds the corpus, and the columns §8 keeps do not include
# research_orgs — so read just the two columns this map needs, straight from the parquet
# (cheap, and it cannot clobber §8's `corpus`).
_papers_all = filter_analysis_window(pd.read_parquet(
    P.SHOWCASE_PLUS, columns=["id", "year", "date", "research_orgs"]
))

_pid_counter = Counter(p for ps in df_ct["pids"] for p in ps)
papers_geo = _papers_all[_papers_all["id"].isin(_pid_counter)].copy()

# ct_country_iso_counts() only needs a `research_orgs` column of {country_code, country_name}
# dicts — the papers frame has exactly that shape, so the SAME function serves both panels
# (and the same ISO-3 matcher, so the maps are comparable).
paper_iso_counts = ct_country_iso_counts(papers_geo)

_papers_have_country = papers_geo["research_orgs"].apply(
    lambda x: any(isinstance(d, dict) and d.get("country_name") for d in parse_listcol(x)))
print(f"Cited UKB papers : {len(papers_geo)}  |  with >=1 research-org country: "
      f"{int(_papers_have_country.sum())}/{len(papers_geo)}  ({paper_iso_counts.size} countries)")
print(f"Trials           : {len(df_ct)}  |  with >=1 research-org country: "
      f"{int(iso_counts.sum() > 0) and int(_has_country.sum())}/{len(df_ct)}"
      f"  ({iso_counts.size} countries)")

fig, axes = plt.subplots(2,1, figsize=(10, 7))
plot_ct_country_map(iso_counts, "  a. Where the UKB-citing trials are run",
                    label="Number of trials", annotate_top=0, ax=axes[0])
plot_ct_country_map(paper_iso_counts, "  b. Where the cited UKB papers were produced",
                    label="Number of papers", annotate_top=0, ax=axes[1])
fig.tight_layout()
savefig(fig, "ct_country_maps_trials_vs_papers")
plt.show()

# Ranked by TRIALS (so the trials column is monotonic and complete — the earlier
# cited-papers sort made this read like a broken raw list). `ratio` = cited papers per trial;
# a high ratio is the "exports influence" signal (research produced here, trials run elsewhere).
_geo = (pd.concat([iso_counts.rename("trials"),
                   paper_iso_counts.rename("cited papers")], axis=1)
          .fillna(0).astype(int))
import numpy as np
_geo["papers/trial"] = np.where(_geo["trials"] > 0,
                                (_geo["cited papers"] / _geo["trials"]).round(1), np.nan)
print("Top 12 countries by TRIALS — trials vs cited papers (research-org basis):")
display(_geo.sort_values("trials", ascending=False).head(12))
print("Top 12 countries by CITED PAPERS (who the trials draw their evidence from):")
display(_geo.sort_values("cited papers", ascending=False).head(12))

In [ ]:

# (2) sector breakdown
cs_mat = country_sector_table(df_ct, top_countries=10)
plot_stacked_sectors(cs_mat, "",
                     "Fractional trial weight for top 10 countries and territories",
                     name="ct_where_sector")
plt.show()

# overall sector engagement (trials involving >=1 org of each sector) — the industry vs
# academia headline for the write-up
sector_involvement = pd.Series(
    {s: int(df_ct["org_sectors"].apply(lambda L: s in L).sum()) for s in SECTOR_ORDER}
).sort_values(ascending=False)
print("Trials involving >=1 organisation of each sector (can overlap):")
print(sector_involvement.to_string())


## 8. The UK Biobank papers behind the trials

The **impact core** of this notebook. Each trial's `publication_ids` are matched against the
UK Biobank corpus (`data/df_dimensions.xlsx`); the intersection is the set of **UKB papers
the trials cite**. We persist that matched set as a standalone dataset and explore what
those papers are about.

The matched frame is written to
`data/non_academic/clinic_trials/ct_ukbb_papers.csv` (one row per UKB paper, with
`n_trials` = how many trials reference it).

In [ ]:
# =============================================================================
# LOAD the UKB corpus — data/showcase/showcase+/showcase_plus_all_endpoints_wide.parquet
# =============================================================================
# This export (unlike papers_20260713.pkl) carries the field-normalised citation ratios
# (RCR / FCR) and `concepts_scores`, which the earlier pickle had dropped. Parquet reads a
# column subset natively, so no xlsx-style caching is needed.
#
# Two shape notes about this file:
#   * list/dict columns are JSON strings — ast.literal_eval (parse_listcol) handles them,
#     verified across every column we touch;
#   * there is no flat `journal.title`; the journal is a JSON blob {'id','title'}, so we
#     derive `journal.title` from it and keep the downstream code unchanged.
import hashlib
DIM_PARQUET = P.SHOWCASE_PLUS
DIM_COLS = ["id", "date", "title", "abstract", "mesh_terms", "category_for_2020", "category_for",
            "concepts", "concepts_scores", "year", "times_cited",
            "relative_citation_ratio", "field_citation_ratio", "type", "journal"]


def load_ukbb_corpus(path=DIM_PARQUET, cols=DIM_COLS):
    d = filter_analysis_window(pd.read_parquet(path, columns=cols))
    # flatten the journal blob -> the `journal.title` column the rest of the notebook uses
    d["journal.title"] = d["journal"].apply(lambda x: parse_dictcol(x).get("title"))
    return d


corpus = load_ukbb_corpus()
print(f"UKB corpus loaded: {len(corpus):,} papers  "
      f"(RCR present for {corpus['relative_citation_ratio'].notna().sum():,})")

# how many trials reference each UKB paper
pid_counter = Counter(p for ps in df_ct["pids"] for p in ps)
ct_papers = corpus[corpus["id"].isin(pid_counter)].copy()
ct_papers["n_trials"] = ct_papers["id"].map(pid_counter)
add_for_columns(ct_papers, "category_for_2020")
ct_papers = ct_papers.sort_values("n_trials", ascending=False).reset_index(drop=True)

OUT = P.CT_UKBB_PAPERS
ct_papers.to_csv(OUT, index=False)

# how many UKB papers each TRIAL cites (its referenced papers that are in our corpus)
corpus_ids = set(corpus["id"])
df_ct["n_ukbb"] = df_ct["pids"].apply(lambda ps: sum(p in corpus_ids for p in ps))

print(f"Matched UKB papers referenced by trials : {len(ct_papers)}")
print(f"Trials referencing >=1 matched UKB paper : {(df_ct['n_ukbb'] > 0).sum()}/{len(df_ct)}")
print(f"Median citations of matched papers       : {ct_papers['times_cited'].median():.0f}")
print(f"Saved -> {P.raw_path(OUT)}")

In [ ]:
df = filter_analysis_window(pd.read_parquet(P.SHOWCASE_PLUS))


In [ ]:
df.loc[df['clinical_trial_ids'].notna()][['id','title','clinical_trial_ids']]

In [ ]:
df_ct.loc[df_ct['id']=='NCT04369807']

In [ ]:
ct_papers['times_cited'].describe()

In [ ]:
print("Number of clinical trials with relative citation ratio > 1:", len(ct_papers.loc[ct_papers['relative_citation_ratio']>1]))
print("Number of clinical trials with more than one citation:", len(ct_papers.loc[ct_papers['n_trials']>1]))

In [ ]:
ct_papers["n_trials"].hist(bins=np.arange(1, ct_papers["n_trials"].max() + 2) - 0.5,
                           color=STYLE["c_primary"], edgecolor=STYLE["edgecolor"], linewidth=0.5)
set_title(plt.gca(), "UKB papers cited by clinical trials", fontsize=STYLE["title_fs"], fontweight="bold")
plt.xlabel("Number of trials citing paper", fontsize=STYLE["label_fs"])
plt.ylabel("Number of UKB papers", fontsize=STYLE["label_fs"])
plt.tight_layout(); 
plt.show()

### 8.1 What are those papers about? — research fields & concepts

Two fractional bar charts over the matched UKB papers:

* **(a) Research fields** — the FOR **L4** taxonomy (L2 was too coarse to be interesting);
* **(b) Concepts** — Dimensions' extracted key phrases.

**Both are counted fractionally**: each paper spreads a total weight of **1** across its
distinct items, so a paper tagged with 4 fields (or 70 concepts) cannot outweigh one tagged
with a single field. The column then sums to the number of papers carrying ≥1 item, and the
two panels are on the same footing.

> **Note on `concepts` in this export.** `papers_20260713.pkl` ships `concepts` as a plain
> **list of strings with no relevance scores** (`concepts_scores` is gone), so the earlier
> relevance-weighted ranking is no longer possible. Papers carry a **median of ~70 concepts**
> each, most of them corpus scaffolding — unfiltered, the ranking is just *study · risk ·
> disease · association · participants · biobank*. So we (i) drop generic phrases via the
> explicit `CONCEPT_STOP` list, and (ii) count fractionally, which is what keeps the
> 70-concept papers from swamping the focused ones. Matching is **exact** on the lowercased
> phrase, so bare *"disease"* is dropped while *"cardiovascular disease"* survives.

In [ ]:
ct_papers['concepts_scores']

In [ ]:
import textwrap

# The concept stop-list + cleaner live in src/utils/ct_concepts.py, shared with §3.6 so the
# two sections cannot drift apart. Three families (stats / corpus / epidemiology) are kept
# separate in that module so they are easy to audit and extend. Matching is EXACT on the
# lowercased phrase, so bare "disease" is dropped while "cardiovascular disease" survives.
from utils.data_analysis_04_non_academic_clinical_trials_concepts import CONCEPT_STOP, clean_concepts

ct_papers["concepts_clean"] = ct_papers["concepts"].apply(clean_concepts)

# ---- fractional counts: each paper spreads weight 1 across its distinct items ----
for_frac = count_items_frac(ct_papers["for_l4"]).head(10)
con_frac = count_items_frac(ct_papers["concepts_clean"]).head(10)

n_for = int(ct_papers["for_l4"].apply(len).gt(0).sum())
n_con = int(ct_papers["concepts_clean"].apply(len).gt(0).sum())
med_con = ct_papers["concepts"].apply(lambda x: len(parse_listcol(x))).median()
print(f"FOR L4   : {n_for}/{len(ct_papers)} papers carry >=1 field")
print(f"Concepts : {n_con}/{len(ct_papers)} papers carry >=1 concept after the stop-list "
      f"(median {med_con:.0f} raw concepts per paper, {len(CONCEPT_STOP)} phrases stop-listed)")

# wrap long labels so neither axis eats the figure
WRAP = 30
for s in (for_frac, con_frac):
    s.index = [x if len(x) <= WRAP else textwrap.fill(x, width=WRAP) for x in s.index]

cstyle = {**STYLE, "title_fs": 13, "label_fs": 11, "annot_fs": 9}
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

plot_hbar(for_frac, "",
          f"Fractional paper weight — research fields (FOR L4, n={n_for})",
          style=cstyle, color=STYLE["c_green"], fmt="{:.1f}", ax=axes[0])
set_title(axes[0], "a.", loc="left", fontsize=cstyle["title_fs"], fontweight="bold")

plot_hbar(con_frac, "",
          f"Fractional paper weight — concepts (n={n_con})",
          style=cstyle, color=STYLE["c_primary"], fmt="{:.2f}", ax=axes[1])
set_title(axes[1], "b.", loc="left", fontsize=cstyle["title_fs"], fontweight="bold")

fig.tight_layout(w_pad=3); savefig(fig, "ct_ukbb_papers_fields_concepts")
plt.show()

print("\nTop fields  :", ", ".join(f"{k.replace(chr(10), ' ')} ({v:.1f})"
                                  for k, v in for_frac.head(5).items()))
print("Top concepts:", ", ".join(f"{k.replace(chr(10), ' ')} ({v:.2f})"
                                 for k, v in con_frac.head(5).items()))

#### 8.1.1 What is each field *about*? — concept profile per FOR L4

The bars above say *how much* of the cited corpus sits in each field; they don't say what
those papers are **about**. This pairs the same L4 ranking (left, identical order) with a
**heatmap sharing the y-axis** (right): each row is a field, each column one of the **top-10
concepts**, and the cell is the **frequency of that concept among that field's papers** —
i.e. the % of Genetics papers that mention *loci*, the % of Oncology papers that mention
*cancer*, and so on. `Spectral_r` runs cool (rare) → warm (common).

A shared column vocabulary is what makes it a matrix, so the columns are the **top 10
concepts overall**. Each field's *own* top-10 (which is what you'd read off a per-field list)
is printed as a table underneath, since those lists differ from row to row.

> **Row `n` matters.** The percentage is over the papers *in that row*, and the tail fields
> are small — *Oncology and Carcinogenesis* is 89% "cancer" from **9 papers**, *Reproductive
> Medicine* from **6**. The paper count is in each y-label; treat the bottom rows as
> indicative, not as rates. Papers are multi-label, so a paper tagged with 3 L4 fields
> contributes to all 3 rows.

In [ ]:
# =============================================================================
# FIELD x CONCEPT — what each FOR L4 field is actually about
# =============================================================================
# Rows  : the same top-10 L4 fields as panel (a), same order (shared y-axis)
# Cols  : the top-10 concepts overall (a shared vocabulary is what makes it a matrix)
# Cell  : % of THAT field's papers whose concepts include the concept
TOP_FIELDS, TOP_CONCEPTS = 10, 10

field_frac = count_items_frac(ct_papers["for_l4"]).head(TOP_FIELDS)
fields = list(field_frac.index)
concepts = list(count_items_frac(ct_papers["concepts_clean"]).head(TOP_CONCEPTS).index)

# papers belonging to each field (multi-label: a paper counts in every field it carries)
field_papers = {f: ct_papers[ct_papers["for_l4"].apply(lambda L: f in L)] for f in fields}

heat = pd.DataFrame(
    [[100 * sub["concepts_clean"].apply(lambda C: c in C).mean() for c in concepts]
     for f, sub in field_papers.items()],
    index=fields, columns=concepts)

# y labels carry the row's paper count — the tail fields are small and the % is over them
ylabels = [textwrap.fill(f, 26) + f"\n(n={len(field_papers[f])})" for f in fields]

# ---- draw: bars (left) + heatmap (right), sharing the y-axis --------------------------
# Both panels use numeric y positions in ASCENDING order (largest field on top), so the
# shared axis lines up row-for-row with panel (a).
n = len(fields)
y = np.arange(n)
vals = field_frac.values[::-1]
M = heat.values[::-1, :]
ylabels_r = ylabels[::-1]

fig, (ax0, ax1) = plt.subplots(
    1, 2, figsize=(12, 7), sharey=True, layout="constrained",
    gridspec_kw={"width_ratios": [0.9, 1.1]})

bars = ax0.barh(y, vals, color=STYLE["c_primary"], edgecolor=STYLE["edgecolor"], linewidth=1)
for b, v in zip(bars, vals):
    ax0.text(v + vals.max() * 0.02, b.get_y() + b.get_height() / 2, f"{v:.1f}",
             va="center", fontsize=STYLE["annot_fs"] - 1)
ax0.set_yticks(y)
ax0.set_yticklabels(ylabels_r, fontsize=STYLE["tick_fs"] - 1)
ax0.text(10, -2.1, "Fractional paper weight" ,fontsize=STYLE["label_fs"])

ax0.set_ylabel("FOR L4 field (n = papers in that field)", fontsize=STYLE["label_fs"])
set_title(ax0, "a. ",
              loc="left", fontsize=13, fontweight="bold")
ax0.margins(x=0.16)
ax0.set_ylim(-0.6, n - 0.4)

im = ax1.imshow(M, aspect="auto", cmap="Spectral_r", origin="lower", vmin=0,
                extent=(-0.5, len(concepts) - 0.5, -0.5, n - 0.5))
ax1.set_xticks(range(len(concepts)))
ax1.set_xticklabels([textwrap.fill(c, 14) for c in concepts],
                     ha="right", fontsize=STYLE["tick_fs"] - 1,rotation=90)
vmax = M.max()
for i in range(n):
    for j in range(len(concepts)):
        ax1.text(j, i, f"{M[i, j]:.1f}" if M[i, j] > 0 else "-", ha="center", va="center",
                 fontsize=STYLE["annot_fs"] - 2,
                 color="white" if M[i, j] > 0.72 * vmax else "black")
set_title(ax1, "b. ",
              loc="left", fontsize=13, fontweight="bold")
cb = fig.colorbar(im, ax=ax1, pad=0.02, fraction=0.04)
ax1.set_xlabel("Top concepts", fontsize=STYLE["label_fs"])
cb.set_label("% of the field's papers mentioning the concept",
             fontsize=STYLE["label_fs"] - 2)

savefig(fig, "ct_field_concept_heatmap")
plt.show()

# ---- each field's OWN top-5 concepts (the per-field list the heatmap columns can't show)
rows = []
for f, sub in field_papers.items():
    top = count_items_frac(sub["concepts_clean"]).head(5)
    share = {c: 100 * sub["concepts_clean"].apply(lambda C: c in C).mean() for c in top.index}
    rows.append({"FOR L4 field": f, "papers": len(sub),
                 "top 5 concepts (% of the field's papers)":
                     ", ".join(f"{c} ({share[c]:.0f}%)" for c in top.index)})
print("Each field's own top-5 concepts:")
with pd.option_context("display.max_colwidth", 200):
    display(pd.DataFrame(rows))

### 8.2 Citation impact & how intensively trials draw on UK Biobank

Two distributions in one figure: **(left)** raw citations of the cited papers
(`times_cited` — heavily-cited landmark papers); **(right)** how many UKB papers each
*trial* cites. (How many trials re-use each paper is printed below the figure.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# (a) raw citation counts (times_cited), log scale
ax = axes[0]
c = ct_papers["times_cited"].clip(lower=1)
ax.hist(c, bins=np.logspace(0, np.log10(c.max()) + 0.05, 20),
        color=STYLE["c_primary"], edgecolor=STYLE["edgecolor"], linewidth=0.5)
ax.set_xscale("log")
med = ct_papers["times_cited"].median()
ax.axvline(med, ls="--", color=STYLE["c_gold"], linewidth=1.5)

ax.annotate(
    f" median {int(med)}",
    xy=(med, ax.get_ylim()[1] * 0.80),          # arrow points here
    xytext=(med * 6, ax.get_ylim()[1] * 0.72),  # text box position
    fontsize=STYLE["annot_fs"] - 1,
    color="black",
    ha="left",
    va="center",

    bbox=dict(
        boxstyle="round,pad=0.3",
        fc="white",
        ec="black",
        lw=1.2,
        alpha=0.95,
    ),

    arrowprops=dict(
        arrowstyle="-|>",
        color=STYLE["c_gold"],
        lw=1.5,
        connectionstyle="arc3,rad=0.25",   # <-- curved arrow
        shrinkA=5,
        shrinkB=3,
    ),
)

#ax.text(med+10, ax.get_ylim()[1]*0.4, f" median {int(med)}", rotation=90, va="top", ha="left",
#        color=STYLE["c_gold"], fontsize=STYLE["annot_fs"])
ax.set_xlabel("Times cited (log scale)", fontsize=STYLE["label_fs"])
ax.set_ylabel("Number of papers", fontsize=STYLE["label_fs"])
set_title(ax, f"a. Citations of cited UKB papers",
             fontsize=STYLE["title_fs"], fontweight="bold", loc="left")

# (b) UKB papers cited per trial
vc1 = df_ct["n_ukbb"].value_counts().sort_index()
axes[1].bar(vc1.index, vc1.values, color=STYLE["c_gold"],
            edgecolor=STYLE["edgecolor"], linewidth=0.6)
for x, v in zip(vc1.index, vc1.values):
    axes[1].text(x, v, str(int(v)), ha="center", va="bottom", fontsize=STYLE["annot_fs"])
axes[1].set_xlabel("Number of UKB papers cited by a trial", fontsize=STYLE["label_fs"])
axes[1].set_ylabel("Number of trials", fontsize=STYLE["label_fs"])
set_title(axes[1], f"b. UKB papers per trial",
                  fontsize=STYLE["title_fs"], fontweight="bold", loc="left")

axes[1].text(0.34, 0.92, f"Top citing trial: \n{textwrap.fill(df_ct.loc[df_ct['n_ukbb'].idxmax(), 'brief_title'], width=40)} ", 
                transform=axes[1].transAxes, fontsize=STYLE["annot_fs"] - 1, ha="left", va="top",
                bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="black", lw=0.8, alpha=0.92))

# add a curve arrow pointing to the annotated trial box from x=7, y=vc1.loc[7] + 5
# a curved arrow can be drawn using FancyArrowPatch, 
from matplotlib.patches import FancyArrowPatch
arrow = FancyArrowPatch( (5, 130), 
                        (7, 20),
                        connectionstyle="arc3,rad=-0.3", 
                        arrowstyle='-|>', 
                        mutation_scale=15, 
                        color='black', 
                        linewidth=1.2)
axes[1].add_patch(arrow)
axes[1].set_xlim(0.2, vc1.index.max()+0.4)
fig.tight_layout(rect=[0, 0, 1, 0.93]); savefig(fig, "ct_ukbb_papers_impact")
plt.show()
print(f"Trials per UKB paper: {(ct_papers['n_trials']>1).sum()} papers cited by >1 trial "
      f"(max {int(ct_papers['n_trials'].max())}).")

In [ ]:
df_ct.loc[df_ct['n_ukbb']>=4]

In [ ]:
ct_papers.sort_values("n_trials", ascending=False).head(5)[['n_trials','relative_citation_ratio','year','journal.title','title']]

In [ ]:
ct_papers

### 8.3 Timeliness & venues — how fast, and from where

Three disclosures that strengthen the impact / funding case:

* **Publication years** of the cited papers — a young, active evidence base;
* **Time-to-trial lag** (`trial start year − paper year`) — how quickly UKB findings reach
  the clinic. We show only **positive** lags (paper precedes trial); links with lag ≤ 0 are
  excluded and counted in the note below — these are trials whose start date pre-dates a
  paper they reference, most likely because the trial record's reference list was **edited /
  updated after registration**;
* **Top journals** — the calibre of the venues the trials are drawing on.

In [ ]:
# lag: for every (trial, cited UKB paper) pair, trial start year - paper publication year
paper_year = dict(zip(ct_papers["id"], ct_papers["year"]))
lags = []
for _, r in df_ct.iterrows():
    sy = r["start_year"]
    if pd.isna(sy):
        continue
    for p in r["pids"]:
        if p in paper_year and pd.notna(paper_year[p]):
            lags.append(int(sy) - int(paper_year[p]))
lags = pd.Series(lags)
lags_pos = lags[lags > 0]                 # keep only positive lags (paper precedes trial)
n_nonpos = int((lags <= 0).sum())
n_neg = int((lags < 0).sum())

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 6),
    gridspec_kw={
        "width_ratios": [1.2,1,1],
        "wspace": 0.5  # increase horizontal spacing
    }
)

# (a) publication years of the cited papers
yr = ct_papers["year"].dropna().astype(int)
yc = yr.value_counts().sort_index()
axes[0].bar([int(x) for x in yc.index], yc.values, color=STYLE["c_primary"],
            edgecolor=STYLE["edgecolor"], linewidth=0.6)
set_title(axes[0], "a. ", fontsize=STYLE["title_fs"], fontweight="bold", loc="left")
axes[0].set_xlabel("Publication year of cited UKB papers"); axes[0].set_ylabel("Papers")
axes[0].set_xlim(ANALYSIS_START_YEAR - 0.9, ANALYSIS_END_YEAR + 1)
axes[0].set_xticks(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1, 2))
#axes[0].set_xticklabels(range(2012, max(yr) + 1), ha="right", fontsize=STYLE["label_fs"] - 1)

# (b) time-to-trial lag — POSITIVE lags only (paper precedes trial start)
axes[1].hist(lags_pos, bins=range(1, int(lags_pos.max()) + 2),
             color=STYLE["c_gold"], edgecolor=STYLE["edgecolor"], linewidth=0.5, align="left")
axes[1].axvline(lags_pos.median(), ls="--", color=STYLE["c_primary"], linewidth=1.5)

axes[1].annotate(
    f" median {lags_pos.median():.0f} yr ",
    xy=(lags_pos.median(), axes[1].get_ylim()[1] * 0.80),          # arrow points here
    xytext=(lags_pos.median() * 1.2, axes[1].get_ylim()[1] * 0.92),  # text box position
    fontsize=STYLE["annot_fs"] +3,
    color="black",
    ha="left",
    va="center",

    bbox=dict(
        boxstyle="round,pad=0.3",
        fc="white",
        ec="black",
        lw=1.2,
        alpha=0.95,
    ),

    arrowprops=dict(
        arrowstyle="-|>",
        color=STYLE["c_primary"],
        lw=1.5,
        connectionstyle="arc3,rad=-0.25",   # <-- curved arrow
        shrinkA=5,
        shrinkB=3,
    ),
)

# add a curved arrow pointing to the median line

#axes[1].text(lags_pos.median(), axes[1].get_ylim()[1]*0.93, f" median {lags_pos.median():.0f} yr",
#             color=STYLE["c_gold"], fontsize=STYLE["annot_fs"])
set_title(axes[1], "b.", fontsize=STYLE["title_fs"], fontweight="bold", loc="left")
axes[1].set_xlabel("Paper to trial lag years (positive only)"); axes[1].set_ylabel("Trial–paper links count")
#axes[1].text(0.97, 0.72, f"excluded (lag ≤ 0): {n_nonpos}\n  of which lag < 0: {n_neg}",
#             transform=axes[1].transAxes, ha="right", va="top",
#             fontsize=STYLE["annot_fs"] - 1, color="#555",
#             bbox=dict(boxstyle="round,pad=0.3", fc="#F4F4F4", ec="#CCC"))

# (c) top journals
jc = ct_papers["journal.title"].value_counts().head(10).sort_values()
axes[2].barh([textwrap.fill(j if 'JAMA' not in j else 'JAMA', width=20) for j in jc.index.astype(str)], jc.values, color=STYLE["c_green"],
             edgecolor=STYLE["edgecolor"], linewidth=0.6)
for i, v in enumerate(jc.values):
    axes[2].text(v + 0.3, i, str(int(v)), va="center", fontsize=STYLE["annot_fs"])
set_title(axes[2], "c. ", fontsize=STYLE["title_fs"], fontweight="bold", loc="left")
axes[2].set_ylabel("Top journals of cited UKB papers");
axes[2].set_xlabel("Papers")

pos1 = axes[1].get_position()
axes[1].set_position([
    pos1.x0 - 0.025,  # move panel b left
    pos1.y0,
    pos1.width,
    pos1.height,
])


fig.tight_layout()
savefig(fig, "ct_papers_timeliness")
plt.show()
print(f"Paper→trial lag (positive only): median {lags_pos.median():.0f} yr, "
      f"mean {lags_pos.mean():.1f} yr over {len(lags_pos)} links.")
print(f"Excluded {n_nonpos} links with lag <= 0 ({n_neg} strictly negative) — trials whose "
      f"start date pre-dates a referenced paper, likely from post-registration edits.")

### 8.4 Title word cloud + the most-referenced papers

In [ ]:
from wordcloud import STOPWORDS
extra_stop = {"biobank", "uk", "study", "using", "analysis", "based", "cohort",
              "association", "associations", "risk", "among", "data"}
stop = set(map(str.lower, STOPWORDS)) | extra_stop
words = Counter()
for t in ct_papers["title"].dropna():
    for w in re.findall(r"[A-Za-z][A-Za-z\-]{2,}", t.lower()):
        if w not in stop:
            words[w] += 1
plot_wordcloud(words, "Titles of the UKB papers cited by clinical trials",
               name="ct_ukbb_paper_titles")
plt.show()

print("Most-referenced UKB papers (by number of citing trials):")
display(ct_papers.head(10)[["n_trials", "relative_citation_ratio", "times_cited",
                            "year", "journal.title", "title"]]
        .rename(columns={"relative_citation_ratio": "RCR", "journal.title": "journal"})
        .reset_index(drop=True))

In [ ]:
df_ct['country']=df_ct['research_orgs'].apply(lambda x: [o.get('country_name') for o in parse_listcol(x) if isinstance(o, dict) and o.get('country_name')])

In [ ]:
df_ct_uk =df_ct.loc[df_ct['country'].apply(lambda x: 'United Kingdom' in x if isinstance(x, list) else False)]

In [ ]:
df_ct_uk['start_year'] = df_ct_uk['start_year'].astype(int)
df_ct_uk['start_year'].value_counts().reindex(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1), fill_value=0).plot(kind='bar', color=STYLE["c_primary"], edgecolor=STYLE["edgecolor"], linewidth=0.5)
# put annotations on the top of each bar with the count
for i, v in enumerate(df_ct_uk['start_year'].value_counts().reindex(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1), fill_value=0).values):
    plt.text(i, v + 0.1, str(v), ha='center', va='bottom', fontsize=STYLE["annot_fs"] - 1)
set_title(plt.gca(), "UK-based trials by start year", fontsize=STYLE["title_fs"], fontweight="bold")
plt.xlabel("Start year", fontsize=STYLE["label_fs"])
plt.ylabel("Number of trials", fontsize=STYLE["label_fs"])
plt.tight_layout()

In [ ]:
df_ct_uk['start_year'] = df_ct_uk['start_year'].astype(int)
df_ct_uk['start_year'].value_counts().reindex(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1), fill_value=0).plot(kind='bar', color=STYLE["c_primary"], edgecolor=STYLE["edgecolor"], linewidth=0.5)
# put annotations on the top of each bar with the count
for i, v in enumerate(df_ct_uk['start_year'].value_counts().reindex(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1), fill_value=0).values):
    plt.text(i, v + 0.1, str(v), ha='center', va='bottom', fontsize=STYLE["annot_fs"] - 1)
set_title(plt.gca(), "UK-based trials by start year", fontsize=STYLE["title_fs"], fontweight="bold")
plt.xlabel("Start year", fontsize=STYLE["label_fs"])
plt.ylabel("Number of trials", fontsize=STYLE["label_fs"])
plt.tight_layout()

---

## Execution boundary before Part 2: Patent impact

The following reset deliberately reproduces the fresh Python kernel used by the former
`04_non_academic_02_patents.ipynb` while leaving files written by earlier parts available to this one.


In [ ]:
# Reproduce the clean-kernel boundary that separated the source notebooks.
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass

import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()

from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part II: Patent impact

This part performs comprehensive patent analysis using modularized functions from `utils/patent_utils.py`. 
It covers:
- Data loading and preparation
- Abstract cleaning
- Country assignment from assignees
- Topic parsing and hierarchical classification
- LLM-based classification (using pre-computed results to skip re-running)
- Summary statistics and visualizations
- Data export

**Note**: LLM classification results are loaded from saved checkpoints, so no actual LLM calls are made in this part.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ast
import json
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Make `src` importable and anchor all paths on the repo root, regardless of the
# directory this notebook is launched from (repo root, src/, or src/data_analysis/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()
from utils.shared_style import apply_typography, set_title
apply_typography()
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, filter_analysis_window
P.FIG_PATENT.mkdir(parents=True, exist_ok=True)

# Import modularized patent-analysis functions.
from utils import shared_patent_utils as patent

# Define color scheme for visualizations
colors_scheme = ["#B80C09", "#D4AF37", "#6E8B3D", "#345995"]

# Set plot style
plt.rcParams['figure.dpi'] = 300

## 1. Load Patent Data

Load the main patent dataset and reference data (UK Biobank papers).

In [ ]:
# Load patent data.
#
# SOURCE REPOINTED 2026-09-20 (D43); the analysis-window filter is kept from the 2026-09-21
# pass. This read `P.PATENTS_DETAILED` — a separate, earlier patent query returning 513
# records. The corpus carries the endpoint pipeline's OWN patent records, collected in the
# same run as the publication records: 767 linked patents, a strict superset, at one index
# date. 24 of the shared 513 had already drifted on `legal_status`, the field this notebook
# is most about.
#
# `endpoint_records` returns the full patent record with the same column names and cell
# shapes the pull had (`assignee_countries` as {id, name} dicts, `category_for_2020` as a
# list of dicts), so every downstream section works unchanged.
from utils.shared_showcase import endpoint_records, load_showcase

df_patent_linked = endpoint_records("patents")
df_patent = filter_analysis_window(
    df_patent_linked, year_col="publication_year", date_col="publication_date"
)
print(f"Patent source: {P.raw_path(P.SHOWCASE_PLUS)} (patents__* endpoint block)")
print(f"✓ Patents linked: {len(df_patent_linked)}; "
      f"in {ANALYSIS_START_YEAR}–{ANALYSIS_END_YEAR}: {len(df_patent)}")

# UK Biobank paper corpus — the showcase-plus parquet.
# Only `id` (citation matching) and a few display columns are needed downstream.
df_all_ukbb = load_showcase(columns=["id", "title", "year", "times_cited"])

print(f"✓ UK Biobank papers loaded: {len(df_all_ukbb)} papers")
print(f"\nPatent data shape: {df_patent.shape}")
print(f"Missing abstracts: {df_patent['abstract'].isnull().sum()}")
print(f"\nSample columns: {df_patent.columns.tolist()[:10]}")

## 2. Data Preparation and Cleaning

Clean patent abstracts and prepare data for analysis.

In [ ]:
# Clean abstracts using modularized function
df_patent["abstract_clean"] = df_patent["abstract"].apply(patent.clean_patent_abstract)

# Prepare publication dates
df_patent['publication_date'] = pd.to_datetime(df_patent['publication_date'], errors='coerce')
df_patent['publication_year'] = df_patent['publication_date'].dt.year
df_patent['year'] = df_patent['publication_year']

print(f"✓ Abstracts cleaned: {df_patent['abstract_clean'].notna().sum()} non-null abstracts")
print(f"✓ Publication dates processed: {df_patent['publication_year'].notna().sum()} valid dates")
print(f"\nYear range: {df_patent['publication_year'].min():.0f} - {df_patent['publication_year'].max():.0f}")

## 3. Country Assignment from Assignees

Assign country information to patents using patent assignee metadata.

In [ ]:
# Assign country information using modularized function
# Use aggressive mode to include rule-based inference
df_with_iso = patent.assign_assignee_countries_with_iso(
    df_patent,
    assignee_col='assignees',
    assignee_names_col='assignee_names',
    use_low_confidence=True,
    primary_pick='most_common'
)

print(f"✓ Country assignment completed")
print(f"✓ Patents with country info: {df_with_iso['assignee_countries'].notna().sum()}")
print(f"\nTop 10 countries:")
all_countries = []
for countries in df_with_iso['assignee_countries']:
    if isinstance(countries, list):
        all_countries.extend(countries)

country_counts = Counter(all_countries)
for country, count in country_counts.most_common(10):
    print(f"  {country}: {count}")


## 4. Topic Parsing and Hierarchical Classification

Parse patent topics and organize them hierarchically by FOR (Field of Research) codes.

In [ ]:
# Parse patent topics using modularized function
df_with_iso = patent.parse_patent_topics(df_with_iso, category_col='category_for')

print(f"✓ Topics parsed for patents")
print(f"✓ Patents with topic data: {df_with_iso['topics_list'].notna().sum()}")

# Calculate topic statistics
df_with_iso['n_topics'] = df_with_iso['topics_list'].apply(len)
avg_topics = df_with_iso['n_topics'].mean()
median_topics = df_with_iso['n_topics'].median()

print(f"\nTopic statistics:")
print(f"  Average topics per patent: {avg_topics:.2f}")
print(f"  Median topics per patent: {median_topics:.0f}")
print(f"  Max topics in a patent: {df_with_iso['n_topics'].max()}")


In [ ]:
# Visualize topic distribution
patent.plot_topics_histogram(
    df_with_iso,
    n_topics_col='n_topics',
    colors=colors_scheme[3],
    figsize=(10, 6),
    savefile=None
)

In [ ]:
# Build top-level topic counts (fractional counting)
topic_counter = Counter()
for topics in df_with_iso['topics_list']:
    for t in topics:
        topic_counter[t] += 1

topic_df = (
    pd.DataFrame(topic_counter.items(), columns=['topic', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"✓ Found {len(topic_df)} unique topics")

# Visualize top topics
patent.plot_top_topics_horizontal(
    topic_df,
    topic_col='topic',
    count_col='count',
    top_n=20,
    colors=[colors_scheme[3]],
    figsize=(12, 8),
    savefile=None  # P.FIG_PATENT / 'top_topics.png'
)

## 5. Geographic, Topic, and Network Analysis

Explore country patterns, topic hierarchies, co-occurrence structure, and topic diversity.

In [ ]:
# Geographic analysis
country_df = patent.build_country_count(df_with_iso)
print(f"✓ Countries with patent counts: {len(country_df)}")
print(f"✓ Total assignee-country occurrences: {int(country_df['count'].sum())}")

patent.plot_bar_matplotlib(country_df, top_n=15, colors=colors_scheme[3], figsize=(10, 6), savefile=None)

In [ ]:
# Topic hierarchy and frequency analysis
df_with_iso['top_level_topics'] = df_with_iso['category_for'].apply(patent.safe_parse_category).apply(patent.collapse_to_top_level)
df_with_iso['topic_count'] = df_with_iso['top_level_topics'].apply(lambda topics: len(topics) if isinstance(topics, list) else 0)

topcode_to_label = {}
for topics in df_with_iso['top_level_topics']:
    if isinstance(topics, list):
        for code, label in topics:
            if code and label and code not in topcode_to_label:
                topcode_to_label[code] = label

print(f"✓ Patents with hierarchical topic data: {df_with_iso['top_level_topics'].notna().sum()}")
print(f"✓ Average topics per patent: {df_with_iso['topic_count'].mean():.2f}")
print(f"✓ Median topics per patent: {df_with_iso['topic_count'].median():.0f}")

patent.plot_topics_histogram(
    df_with_iso,
    n_topics_col='topic_count',
    colors=colors_scheme[3],
    figsize=(10, 6),
    savefile=None,
)

topic_counter = Counter()
for topics in df_with_iso['top_level_topics']:
    if isinstance(topics, list):
        for code, _ in topics:
            if code:
                topic_counter[code] += 1

topic_df = (
    pd.DataFrame(topic_counter.items(), columns=['topic', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
print(f"✓ Found {len(topic_df)} unique top-level topics")

patent.plot_top_topics_horizontal(
    topic_df,
    topic_col='topic',
    count_col='count',
    top_n=20,
    colors=[colors_scheme[3]],
    figsize=(12, 8),
    savefile=None,
)

In [ ]:
# Topic co-occurrence and sustainability analysis
single_topic_patents = df_with_iso[df_with_iso['topic_count'] == 1]
multi_topic_patents = df_with_iso[df_with_iso['topic_count'] >= 2]

print(f"Single-topic patents: {len(single_topic_patents)} ({len(single_topic_patents) / len(df_with_iso) * 100:.1f}%)")
print(f"Multi-topic patents: {len(multi_topic_patents)} ({len(multi_topic_patents) / len(df_with_iso) * 100:.1f}%)")

single_topic_counter = Counter()
for topics in single_topic_patents['top_level_topics']:
    if isinstance(topics, list):
        for code, _ in topics:
            if code:
                single_topic_counter[code] += 1

print("\nTop self-sustaining topics:")
for code, count in single_topic_counter.most_common(10):
    print(f"  {code}: {count}")

co_graph, pair_counter = patent.build_topic_cooccurrence_network(
    df_with_iso,
    topics_col='top_level_topics',
    min_weight=5,
)
print(f"\nCo-occurrence graph nodes: {co_graph.number_of_nodes()}")
print(f"Co-occurrence graph edges: {co_graph.number_of_edges()}")

patent.plot_topic_cooccurrence_network(
    co_graph,
    code_to_label=topcode_to_label,
    figsize=(12, 10),
    savefile=None,
)

In [ ]:
# Patent topics by country
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    'custom_cmap',
    [colors_scheme[3], colors_scheme[0]]
)

country_topic_rows, pivot_country_topic, top_countries, top_topic_codes, country_counter, topic_counter_frac = patent.analyze_country_topics(
    df_with_iso,
    topics_col='top_level_topics',
    country_col='assignee_countries',
    code_to_label=topcode_to_label,
    top_countries=8,
    top_topics=8,
)

if not country_topic_rows.empty:
    topic_by_country = (
        country_topic_rows.groupby(['country', 'topic_code', 'topic_label'])['weight']
        .sum()
        .reset_index(name='count')
    )
    print(f"✓ Countries with topic data: {topic_by_country['country'].nunique()}")
    print(f"✓ Topics in country matrix: {topic_by_country['topic_code'].nunique()}")

    patent.plot_country_topic_heatmap(
        pivot_country_topic,
        code_to_label=topcode_to_label,
        figsize=(10, 6),
        savefile=None,
        cmap=cmap
    )

    patent.plot_country_dominant_topics(
        topic_by_country,
        top_countries=top_countries,
        code_to_label=topcode_to_label,
        figsize=(8, 6),
        savefile=None,
    )
else:
    print('No country-topic data available for visualization.')

In [ ]:
# Filing status over time (source-faithful to original notebook)
col = 'legal_status_replaced'
patent.plot_filing_status_over_time(df_with_iso, col, figsize=(10, 6), savefigure=False)

In [ ]:
# Assembled summary visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

# a. Filing status over time
col = 'legal_status_replaced'
patent.plot_filing_status_over_time(df_with_iso, col, figsize=(10, 6), savefigure=False, ax=axes[0, 0], title='a. Patent counts by filing status and year')

# b. Top countries by assignee occurrences
top_country_counts = country_df.sort_values('count', ascending=False).head(10)
axes[0, 1].bar(top_country_counts['iso2'], top_country_counts['count'], color=colors_scheme[3], edgecolor='black', linewidth=0.8)
set_title(axes[0, 1], 'b. Top countries by assignee occurrences', fontsize=11, fontweight='bold', loc='left')
axes[0, 1].set_xlabel('Country')
axes[0, 1].set_ylabel('Count')
axes[0, 1].spines['top'].set_visible(False)
axes[0, 1].spines['right'].set_visible(False)

# c. Distribution of topics per patent
axes[1, 0].hist(df_with_iso['topic_count'], bins=range(1, int(df_with_iso['topic_count'].max()) + 2), edgecolor='black', color=colors_scheme[3])
axes[1, 0].axvline(df_with_iso['topic_count'].mean(), linestyle='--', color=colors_scheme[2], label=f"Mean = {df_with_iso['topic_count'].mean():.2f}")
axes[1, 0].axvline(df_with_iso['topic_count'].median(), linestyle=':', color=colors_scheme[0], label=f"Median = {df_with_iso['topic_count'].median():.0f}")
axes[1, 0].legend(frameon=False)
set_title(axes[1, 0], 'c. Distribution of topics per patent', fontsize=11, fontweight='bold', loc='left')
axes[1, 0].set_xlabel('Number of topics per patent')
axes[1, 0].set_ylabel('Number of patents')
axes[1, 0].spines['top'].set_visible(False)
axes[1, 0].spines['right'].set_visible(False)

# d. Patent topics by country (%)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    'custom_cmap',
    [colors_scheme[3], colors_scheme[0]]
)
patent.plot_country_topic_heatmap(
    pivot_country_topic,
    code_to_label=topcode_to_label,
    figsize=(10, 6),
    savefile=None,
    ax=axes[1, 1],
    cmap=cmap,
    title='d. Patent topics by country (%)',
)

plt.tight_layout()
plt.show()

In [ ]:
# Export showcase data and UKBB citation analysis
showcase_cols = [
    col for col in [
        'id', 'title', 'abstract_clean', 'publication_date', 'publication_year',
        'assignee_country_primary', 'assignee_countries', 'filing_status',
        'category_for', 'topic_count'
    ]
    if col in df_with_iso.columns
]

df_patent_showcase = df_with_iso.copy()
if 'filing_status' in df_patent_showcase.columns:
    filing_status_values = set(df_patent_showcase['filing_status'].dropna().astype(str))
    preferred_statuses = [status for status in ['Application', 'Grant'] if status in filing_status_values]
    if preferred_statuses:
        df_patent_showcase = df_patent_showcase[df_patent_showcase['filing_status'].isin(preferred_statuses)].copy()

if df_patent_showcase.empty:
    df_patent_showcase = df_with_iso.copy()

df_patent_showcase = df_patent_showcase[showcase_cols].drop_duplicates(subset=['id']).copy()
P.PATENT.mkdir(parents=True, exist_ok=True)
showcase_path = P.PATENT / 'df_patent_showcase.csv'
df_patent_showcase.to_csv(showcase_path, index=False)
print(f"Saved showcase subset with {len(df_patent_showcase)} patents to {P.raw_path(showcase_path)}")

if 'publication_ids' in df_with_iso.columns:
    df_with_iso['in_ukbb'] = df_with_iso['publication_ids'].apply(
        lambda x: patent.find_ukbb_papers(x, df_all_ukbb['id'].tolist())
    )

    paper_counter = Counter()
    for papers in df_with_iso['in_ukbb']:
        for paper_id in papers:
            paper_counter[paper_id] += 1

    df_ukbb_cited_papers = (
        pd.DataFrame(paper_counter.items(), columns=['id', 'count'])
        .sort_values(['count', 'id'], ascending=[False, True], kind='stable')
        .reset_index(drop=True)
    )

    if not df_ukbb_cited_papers.empty:
        if 'id' in df_all_ukbb.columns:
            df_ukbb_cited_papers = df_ukbb_cited_papers.merge(df_all_ukbb, on='id', how='left')

        print(f"Found {len(df_ukbb_cited_papers)} UKBB papers cited by patents")
        print(f"Total citation events: {int(df_ukbb_cited_papers['count'].sum())}")

        patent.plot_top_cited_papers(
            df_ukbb_cited_papers,
            title_col='title',
            count_col='count',
            top_n=15,
            colors=[colors_scheme[3]],
            figsize=(14, 8),
            savefile=None,
        )
    else:
        print('No UKBB papers were cited by the current patent set.')
else:
    print('publication_ids column is missing; skipping UKBB citation analysis.')

output_path = P.PATENT / 'patents_modularized_export.csv'
df_with_iso.to_csv(output_path, index=False)
print(f"Saved modularized analysis dataset to {P.raw_path(output_path)}")

# Record which cohort was used. Updated 2026-09-20 (D43): the source is no longer the
# separate detailed pull but the corpus's own patents__* endpoint block, collected in the
# same extraction as the publication records — so the provenance names the corpus, and
# records how many linked patents the analysis window left behind.
_source_stat = P.SHOWCASE_PLUS.stat()
output_path.with_suffix(".provenance.json").write_text(json.dumps({
    "source": P.raw_path(P.SHOWCASE_PLUS),
    "source_block": "patents__*",
    "source_size_bytes": _source_stat.st_size,
    "source_modified_utc": pd.Timestamp(_source_stat.st_mtime, unit="s", tz="UTC").isoformat(),
    "linked_records": len(df_patent_linked),
    "records": len(df_with_iso),
    "analysis_start_date": "2013-01-01",
    "analysis_end_date": "2025-12-31",
    "method": "Recompute country, FOR topic and filing-status columns for the linked patent cohort, restricted to the analysis window",
}, indent=2) + "\n", encoding="utf-8")


---

## Execution boundary before Part 3: Policy and Altmetric attention

The following reset deliberately reproduces the fresh Python kernel used by the former
`04_non_academic_03_altmetric.ipynb` while leaving files written by earlier parts available to this one.


In [ ]:
# Reproduce the clean-kernel boundary that separated the source notebooks.
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass

import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()

from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part III: Policy and Altmetric attention

This compact part joins policy and Altmetric observations to the eligible Showcase+
publication cohort, reports source coverage explicitly, and retains the original impact
scatter, annual inset, summary statistics, and annotated papers.


In [ ]:
import sys
from pathlib import Path

# Anchor every path on the repo root regardless of where the notebook is launched
# from (repo root, src/, or src/data_analysis/). See shared_paths for the why.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, filter_analysis_window

# --- project plotting style ------------------------------------------------------
# This notebook used to set `plt.rcParams['font.family'] = 'Helvetica'` by hand -- the
# only notebook in the project that picked its own font. The palette, font sizes and
# output directory now come from universal_settings.yml like everywhere else.
from utils.shared_style import PNG_DPI, apply_style, extended_palette, load_style, savefig, use_style
from utils.shared_style import apply_typography, finalize_figure
apply_typography()
STYLE = load_style("04_non_academic_03_altmetric")

# --- the corpus ------------------------------------------------------------------
# Was `pd.read_excel('../data/dimensions/api/raw/combined/202511/df_dimensions.xlsx')`,
# a file that no longer exists. load_showcase reads the showcase-plus parquet and parses
# the nested columns as JSON -- `authors` arrives as a real list of dicts, so the
# ast.literal_eval branch in get_first_author_from_row() is now dead code, not a
# silent []-returning trap. See src/utils/shared_showcase.py.
from utils.shared_showcase import load_showcase


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
import ast

# ================================================================
# 1. LOAD BOTH DATASETS
# ================================================================

# Prefer a real Altmetric export; fall back to the table rebuilt from the corpus
# and the policy pull. Same column names either way, so nothing below changes --
# but the two are NOT equivalent, and the printed banner says how.
HAS_NEWS = P.ALTMETRIC_CSV.exists()   # only a real export carries news mentions
if HAS_NEWS:
    df_alt = pd.read_csv(P.ALTMETRIC_CSV)
    print(f"Altmetric source: real export, {len(df_alt):,} rows")
else:
    from utils import data_analysis_04_non_academic_altmetric_from_corpus as ALT
    df_alt = pd.read_csv(P.ALTMETRIC_DERIVED) if P.ALTMETRIC_DERIVED.exists() \
        else ALT.build_altmetric_table(out_path=P.ALTMETRIC_DERIVED)
    print("Altmetric source: REBUILT FROM THE CORPUS (no real export on disk)")
    print(" ", ALT.coverage_report(df_alt))
    print("  'News mentions' is 0 for every row because NO SOURCE FOR IT EXISTS in")
    print("  this project -- 0 means unknown, not 'no coverage'. The x axis below is")
    print("  therefore policy citations alone. Do not label it 'news and policy'.")
df_alt = filter_analysis_window(df_alt, year_col="Year", date_col="Publication Date")
df_dim = load_showcase(
    columns=["id", "doi", "year", "date", "times_cited", "journal_title_raw", "authors"],
    parse=["authors"],
)
df_dim = filter_analysis_window(df_dim)

# ================================================================
# 2. CLEAN DOI AND MERGE
# ================================================================

def clean_doi(x):
    if isinstance(x, str):
        return x.strip().lower()
    return np.nan

df_alt["DOI_clean"] = df_alt["DOI"].apply(clean_doi)
df_dim["DOI_clean"] = df_dim["doi"].apply(clean_doi)

df_alt = df_alt.dropna(subset=["DOI_clean"])
df_dim = df_dim.dropna(subset=["DOI_clean"])

# Deduplicate Dimensions: keep most cited entry for each DOI
df_dim = df_dim.sort_values("times_cited", ascending=False)
df_dim = df_dim.drop_duplicates(subset="DOI_clean", keep="first")

# Merge Altmetric + Dimensions
df_merged = df_alt.merge(
    df_dim,
    on="DOI_clean",
    how="inner",
    suffixes=("_altmetric", "_dim")
)

# ================================================================
# 3. PREPARE DATA FOR PLOTTING
# ================================================================

df_plot = filter_analysis_window(df_merged, year_col="year", date_col="Publication Date")
# Attention/citation counts remain the source snapshot totals for these eligible papers.

df_plot["News mentions"] = df_plot["News mentions"].fillna(0)
df_plot["Policy mentions"] = df_plot["Policy mentions"].fillna(0)
df_plot["Altmetric Attention Score"] = df_plot["Altmetric Attention Score"].fillna(0)

df_plot["Total Substantive Mentions"] = (
    df_plot["News mentions"] + df_plot["Policy mentions"]
)

df_plot["Publication Date"] = pd.to_datetime(df_plot["Publication Date"], errors="coerce")
df_plot["Year"] = df_plot["Publication Date"].dt.year

df_scatter = df_plot[
    (df_plot["Total Substantive Mentions"] > 0) &
    (df_plot["Altmetric Attention Score"] > 0)
]

yearly = df_plot.groupby("Year").agg({
    "News mentions": "sum",
    "Policy mentions": "sum"
}).reindex(range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1), fill_value=0)

# ================================================================
# 4. SUMMARY STATISTICS (TOP LEFT BOX)
# ================================================================

n_outputs = len(df_scatter)
mean_aas = df_scatter["Altmetric Attention Score"].mean()
median_aas = df_scatter["Altmetric Attention Score"].median()
max_aas = df_scatter["Altmetric Attention Score"].max()

mean_news = df_scatter["News mentions"].mean()
mean_policy = df_scatter["Policy mentions"].mean()

stats_text = (
    f"Total outputs analysed: {n_outputs}\n"
    f"Mean Altmetric Attention Score: {mean_aas:.1f}\n"
    f"Median Altmetric Attention Score: {median_aas:.1f}\n"
    f"Max Altmetric Attention Score: {max_aas:.0f}\n"
    + (f"Mean 'News' mentions: {mean_news:.1f}\n" if HAS_NEWS
       else "News mentions: no source in this project\n")
    + f"Mean 'Policy' mentions: {mean_policy:.2f}"
)

# ================================================================
# 5. HELPER: EXTRACT FIRST AUTHOR + BUILD ANNOTATION TEXT
# ================================================================

def get_first_author_from_row(row):
    """
    Tries, in order:
    1. Dimensions 'authors' as list of dicts (or JSON string of that).
    2. 'authors' as plain string "Surname, First; S2, F2; ..."
    3. Altmetric 'Authors at my Institution' field.
    """
    first_author = None

    a = row.get("authors", None)

    # 1) authors already a Python list[dict]
    if isinstance(a, list) and len(a) > 0:
        d0 = a[0]
        if isinstance(d0, dict):
            ln = d0.get("last_name") or d0.get("surname") or ""
            fn = d0.get("first_name") or d0.get("given_name") or ""
            first_author = (ln or fn).strip() or None

    # 2) authors as JSON-like string of dicts or a simple string
    if first_author is None and isinstance(a, str) and a.strip():
        s = a.strip()
        if s.startswith("[") and "last_name" in s:
            # JSON-ish list of dicts
            try:
                lst = ast.literal_eval(s)
                if isinstance(lst, list) and lst:
                    d0 = lst[0]
                    if isinstance(d0, dict):
                        ln = d0.get("last_name") or d0.get("surname") or ""
                        fn = d0.get("first_name") or d0.get("given_name") or ""
                        first_author = (ln or fn).strip() or None
            except Exception:
                pass

        if first_author is None:
            # Treat as "Surname, First; S2, F2"
            part = s.split(";")[0]
            if "," in part:
                first_author = part.split(",")[0].strip()
            else:
                first_author = part.split()[0].strip()

    # 3) Fallback: Altmetric 'Authors at my Institution'
    if not first_author:
        a2 = row.get("Authors at my Institution", None)
        if isinstance(a2, str) and a2.strip():
            part = a2.split(";")[0]
            if "," in part:
                first_author = part.split(",")[0].strip()
            else:
                first_author = part.split()[0].strip()

    if not first_author:
        first_author = "Author"

    return first_author


def build_annotation_text(row):
    first_author = get_first_author_from_row(row)

    year = row.get("Year")
    year_str = f"{int(year)}" if pd.notna(year) else "n.d."

    # Prefer Dimensions journal title; fall back to Altmetric
    journal = row.get("journal_title_raw")
    if not isinstance(journal, str) or not journal.strip():
        journal = row.get("Journal/Collection Title", "")
    journal = journal if isinstance(journal, str) else ""

    doi = row.get("DOI", "")
    aas = row["Altmetric Attention Score"]

    cites = row.get("times_cited")
    if pd.isna(cites) and "Number of Dimensions citations" in row:
        cites = row["Number of Dimensions citations"]
    cites_str = "NA" if pd.isna(cites) else f"{int(cites)}"

    txt = (
        f"{first_author} et al. ({year_str})\n"
        f"{journal}\n"
        f"DOI: {doi}\n"
        f"Altmetric Score: {aas:.0f}\n"
        f"Citations: {cites_str}"
    )
    return txt

# ================================================================
# 6. PLOT (MAIN FIGURE)
# ================================================================

fig, ax = plt.subplots(figsize=(16, 9))

ax.scatter(
    df_scatter["Total Substantive Mentions"],
    df_scatter["Altmetric Attention Score"],
    s=30 + 16*np.sqrt(df_scatter["Total Substantive Mentions"]),
    color=(52/255, 89/255, 149/255, 0.35),
    edgecolor="k"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(
    "Total Substantive Mentions (News + Policy)" if HAS_NEWS
    else "Policy citations (Dimensions) \u2014 news mentions unavailable",
    fontsize=16,
)
ax.set_ylabel("Altmetric Attention Score", fontsize=16)

# Statistics box
ax.text(
    0.02, 0.98, stats_text,
    transform=ax.transAxes,
    fontsize=13,
    va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='k', lw=1)
)

# Outward ticks/spines
ax.tick_params(axis="both", which="both", direction="out")
for spine in ax.spines.values():
    spine.set_position(("outward", 10))

# ================================================================
# 7. INSET (YEARLY NEWS + POLICY)
# ================================================================

inset = inset_axes(
    ax,
    width="35%",
    height="30%",
    loc="lower right",
    borderpad=2,
    # Ordinary Axes keeps twinx from reusing the inset's locator artist.
    axes_class=plt.Axes,
)

if HAS_NEWS:
    line_news, = inset.plot(
        yearly.index, yearly["News mentions"], markeredgecolor='k',
        color="#D4AF37", marker="o", linewidth=1.5, markersize=STYLE["marker_size"],
        label="News (left)"
    )
    inset.set_ylabel("Total News Mentions", fontsize=12)
    inset.tick_params(axis="y", labelsize=8)
else:
    line_news = None
    inset.set_yticks([])
    inset.set_ylabel("")

inset.tick_params(axis="both", which="both", direction="out")
for spine in inset.spines.values():
    spine.set_position(("outward", 6))

ax2 = inset.twinx()
line_policy, = ax2.plot(
    yearly.index, yearly["Policy mentions"], markeredgecolor='k',
    color="#B80C09", marker="s", linewidth=1.5, markersize=STYLE["marker_size"],
    label="Policy (right)"
)
ax2.set_ylabel("Total Policy Mentions", fontsize=12)
ax2.tick_params(axis="y", labelsize=8)
ax2.spines["right"].set_position(("outward", 6))

inset.xaxis.set_label_position("top")
inset.xaxis.tick_top()
inset.tick_params(axis="x", bottom=False, labelsize=8)

inset.grid('')
ax2.grid('')

inset.legend(
    handles=[h for h in (line_news, line_policy) if h is not None],
    loc="upper left",
    fontsize=11,
    frameon=True,
    fancybox=True,
    edgecolor='k',
    framealpha=1,
    borderpad=0.4,
    handlelength=1.5,
)

# ================================================================
# 8. ANNOTATE TOP TWO PAPERS (CURVED ARROWS)
# ================================================================

top2 = df_scatter.nlargest(2, "Total Substantive Mentions")

# First: above-left
p1 = top2.iloc[0]
ax.annotate(
    build_annotation_text(p1),
    xy=(p1["Total Substantive Mentions"], p1["Altmetric Attention Score"]),
    xytext=(p1["Total Substantive Mentions"] *.25,
            p1["Altmetric Attention Score"] * .4),
    fontsize=9,
    ha='right',
    va='bottom',
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="k", lw=1),
    arrowprops=dict(
        arrowstyle="-|>",
        color='k',
        lw=1.2,
        connectionstyle="arc3,rad=-0.35"
    )
)

# Second: below-right
p2 = top2.iloc[1]
ax.annotate(
    build_annotation_text(p2),
    xy=(p2["Total Substantive Mentions"], p2["Altmetric Attention Score"]),
    xytext=(p2["Total Substantive Mentions"] * 0.45,
            p2["Altmetric Attention Score"] * 0.1),
    fontsize=9,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="k", lw=1),
    arrowprops=dict(
        arrowstyle="-|>",
        color='k',
        lw=1.2,
        connectionstyle="arc3,rad=0.25"
    )
)

# ================================================================
# 9. MAIN GRID + DESPINE
# ================================================================

ax.grid(True, which="major", linestyle="--", alpha=0.35, zorder=0)
sns.despine(ax=ax)



import matplotlib.lines as mlines

# Example substantive attention values
example_vals = [10, 100, 1000]

# Compute marker sizes using the same rule used in the scatter
example_sizes = [30 + 12*np.sqrt(v) for v in example_vals]

legend_handles = [
    plt.scatter([], [], 
        s=example_sizes[i], 
        color= (52/255, 89/255, 149/255, 0.35), 
        edgecolor=(0,0,0,1),
        label=f"{example_vals[i]} mentions"
    )
    for i in range(len(example_vals))
]

ax.legend(
    handles=legend_handles,
    #title="Circle size shows total\nNews + Policy mentions",
    loc="lower left",
    frameon=True,
    fancybox=True,
    framealpha=1,
    edgecolor="k",
    fontsize=11,
    title_fontsize=12,
    borderpad=0.6,
    ncols=3
)

finalize_figure(plt.gcf())
for _ext, _kw in [("pdf", {}), ("png", {"dpi": PNG_DPI})]:
    plt.savefig(P.FIG_NON_ACADEMIC / f"impact.{_ext}", bbox_inches="tight", **_kw)

---

## Execution boundary before Part 4: Non-academic collaboration

The following reset deliberately reproduces the fresh Python kernel used by the former
`04_non_academic_04_collaboration.ipynb` while leaving files written by earlier parts available to this one.


In [ ]:
# Reproduce the clean-kernel boundary that separated the source notebooks.
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass

import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()

from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part IV: Non-academic collaboration

Load saved organisation classifications, then analyse collaboration by sector,
organisation, year, discipline and geography. A complete classification cache can
rebuild a missing classified table without an API call.

The analysis uses `data/analysis/non_academic/collaboration/non_academic_flagged_full_company.csv`.
If neither labels nor a complete cache are available, it reports missing input data.
Paid classification is disabled by default. To deliberately run it, set
`UKB_ALLOW_CLASSIFICATION=1` and configure the Anthropic client separately.


In [ ]:
import os
import sys
from pathlib import Path

# Anchor every path on the repo root regardless of where the notebook is launched
# from (repo root, src/, or src/data_analysis/). See shared_paths for the why.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, filter_analysis_window

# The helpers module is edited while this notebook stays open. Without
# autoreload, re-running a plotting cell silently reuses the function that
# was imported when the kernel started, so edits appear to have no effect.
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from utils.shared_showcase import load_showcase
from utils import data_analysis_04_non_academic_collab_classifier as CLS
from utils import data_analysis_04_non_academic_collab_helpers as h

h.apply_project_plot_style()

# ---------------------------------------------------------------- run controls
FORCE_RECLASSIFY = False        # True rebuilds the table; cached labels still apply
ALLOW_API_CLASSIFICATION = os.environ.get("UKB_ALLOW_CLASSIFICATION", "").lower() in {"1", "true", "yes"}
CLS_MODEL = CLS.DEFAULT_MODEL   # "claude-sonnet-5" — see the module for why not Opus/Haiku
CLS_EFFORT = CLS.DEFAULT_EFFORT # "low" — short-string classification, not reasoning

# Both live under data/analysis/, with the other analysis-derived tables. The classified
# file is an INPUT to §2 onward, not a deliverable, so it does not belong in output/.
CLASSIFIED_PATH = P.COLLAB_FLAGGED   # data/analysis/non_academic/collaboration/…full_company.csv
CACHE_PATH = P.COLLAB_CACHE          # …/collab_classifier_cache.jsonl

print("classified file:", P.raw_path(CLASSIFIED_PATH), "| exists:", CLASSIFIED_PATH.exists())

## 1. Tag collaborators

`load_or_classify` returns the existing classified file untouched when it is
present. The index lists it produces are **positions in `research_orgs`** — that
alignment is what lets §2 read each organisation's GRID type and country to assign
a sector, so the classifier and the taxonomy must never be pointed at different
organisation lists.

In [ ]:
if not CLASSIFIED_PATH.is_file() and not CACHE_PATH.is_file() and not ALLOW_API_CLASSIFICATION:
    raise FileNotFoundError(
        f"Missing collaborator labels: {P.raw_path(CLASSIFIED_PATH)}; "
        "restore this file or its classification cache. API classification is disabled."
    )

# The corpus. `research_orgs` is the classifier's input AND the taxonomy's lookup
# table, so it has to be parsed here and carried through to the saved file.
# `journal_title_raw` and `research_org_countries` are carried for §7 and §10: without
# the first, _pick_journal_name returns "" for every row and the journal table silently
# comes back empty; the second is §10's fallback when an org carries no GRID country.
# They must be requested HERE, before classification — the file written below is what
# every later section reads.
corpus = load_showcase(
    columns=["id", "title", "year", "date", "times_cited", "altmetric",
             "category_for_2020", "research_orgs", "authors",
             "journal_title_raw", "research_org_countries"],
    parse=["research_orgs", "authors", "category_for_2020"],
)
corpus = filter_analysis_window(corpus)
print(f"corpus: {len(corpus):,} publications")

df_flagged, ran_classifier = CLS.load_or_classify(
    corpus,
    out_path=CLASSIFIED_PATH,
    cache_path=CACHE_PATH,
    force=FORCE_RECLASSIFY,
    allow_api=ALLOW_API_CLASSIFICATION,
    model=CLS_MODEL,
    effort=CLS_EFFORT,
)
print(f"classifier ran this session: {ran_classifier}")

## 2. Prepare and derive the sector taxonomy

`require_sector_columns=False` on purpose: the classifier emits index lists, and
`add_non_academic_sector_taxonomy` derives the eight sector labels from those plus
each organisation's GRID metadata. Requiring the columns to pre-exist would demand
that the LLM name the sectors itself — a second, less reliable judgement where a
lookup does the job.

In [ ]:
df, data_path, raw_df = h.load_and_prepare_dataframe(
    data_path=CLASSIFIED_PATH,
    require_sector_columns=False,
)
# Saved labels can predate the current corpus; retain only eligible corpus IDs.
df = df.loc[df["id"].isin(corpus["id"])].copy()
raw_df = raw_df.loc[raw_df["id"].isin(corpus["id"])].copy()

print(f"Loaded file: {P.raw_path(data_path)}")
print(f"Loaded raw rows: {len(raw_df)}")
print(f"Rows after data preparation: {len(df)}")
print(f"Rows with parsed institutions: {(df['all_institutions'].map(len) > 0).sum()}")
print(f"Rows with any non-academic org parsed: {(df['non_academic_institutions_norm'].map(len) > 0).sum()}")
print(f"Rows with any company parsed: {(df['company_institutions_norm'].map(len) > 0).sum()}")
print(f"Rows with parsed affiliation mentions: {(df['affiliation_names'].map(len) > 0).sum()}")
print(f"Rows with >=1 sector label: {(df['non_academic_sector_labels'].map(len) > 0).sum()}")

if len(df) <= 1000:
    print(
        "WARNING: loaded a small dataset (<=1000 rows). "
        "If unexpected, check input file selection."
    )

## 3. Core Statistics and Top Collaborators

Statistics are computed from taxonomy sectors using paper-level collaborator organization contributions.

In [ ]:
top_sector_tables, unique_counts = h.build_top_sector_collaborator_tables(df, n=30)
stats = h.compute_summary_stats(df, unique_counts=unique_counts)

for line in h.summary_stats_as_lines(stats):
    print(line)

for label in h.NON_ACADEMIC_SECTOR_LABELS:
    table = top_sector_tables.get(label, pd.DataFrame()).head(10)
    if table.empty:
        print(f"Top {label} collaborators: none")
        continue
    display(table.style.set_caption(f"Top {label} collaborators"))


## 4. Collaborator Sector Taxonomy

Sector labels are assigned per collaborator using structured `research_orgs` metadata and classifier indices (academic + non-academic union).

Taxonomy used:
- University/HEI
- Hospital/Clinical
- Government/Public
- Research institute/Centre
- Nonprofit/Charity
- Company (non-UK)
- UK company
- Other/Unknown

In [ ]:
sector_summary_df = h.build_non_academic_sector_summary(df)

print("Collaborator taxonomy summary (derived from research_orgs + index mapping):")
display(
    sector_summary_df.assign(
        mentions_pct=lambda d: d["mentions_pct"].round(1),
        papers_pct_of_taxonomy_papers=lambda d: d["papers_pct_of_taxonomy_papers"].round(1),
    )
)

h.plot_non_academic_sector_breakdown(sector_summary_df, value_col="institution_mentions")
h.plot_non_academic_sector_breakdown(sector_summary_df, value_col="papers")


## 5. Organization-Level Distributions and Rankings

Organization rankings and prevalence figures are shown by taxonomy sector.

In [ ]:
sector_order = [
    "Hospital/Clinical",
    "University/HEI",
    "Government/Public",
    "Research institute/Centre",
    "Nonprofit/Charity",
    "Company (non-UK)",
    "UK company",
    "Other/Unknown",
]
sector_colors = {
    "Hospital/Clinical": "#2A9D8F",
    "University/HEI": "#457B9D",
    "Government/Public": "#8D99AE",
    "Research institute/Centre": "#6A994E",
    "Nonprofit/Charity": "#A1C181",
    "Company (non-UK)": "#5E548E",
    "UK company": "#D4AF37",
    "Other/Unknown": "#BDBDBD",
}

for label in sector_order:
    table = top_sector_tables.get(label, pd.DataFrame())
    print(f"Top {label} collaborators:")
    display(table.head(25))
    if not table.empty:
        h.plot_top_orgs(
            table,
            f"Top {label} collaborators by paper count",
            "Papers",
            label,
            sector_colors.get(label, "#BDBDBD"),
        )

h.plot_collaborator_count_distributions(df)


## 6. Time Trends

Three views are shown:
- cumulative paper counts by taxonomy sector,
- annual share of all papers by taxonomy sector,
- annual share of taxonomy-collaboration papers that include company sectors.

In [ ]:
df = h.ensure_year_column(df)

h.plot_cumulative_by_type(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
h.plot_yearly_share_by_type(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
h.plot_company_share_within_non_academic(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)


## 7. Additional Publication Figures

These figures summarize taxonomy-sector structure and impact:
- annual primary-sector mix in counts and 100% shares,
- overlap heatmap of taxonomy sector flags,
- citation distributions and yearly median citation trajectories by taxonomy groups,
- collaborator concentration curves,
- top-journal non-academic sector share comparisons.

In [ ]:
h.plot_collaboration_mix_stacked_area(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
h.plot_collaboration_mix_share_stacked_area(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
h.plot_flag_overlap_heatmap(df)
h.plot_citation_distribution_by_group(df, citation_col="times_cited")
h.plot_yearly_median_log_citations_by_group(
    df,
    citation_col="times_cited",
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    min_papers_per_point=20,
)
h.plot_collaborator_concentration_curves(df)

journal_company_df = h.build_journal_company_table(df, top_n=20, min_papers=25)
print("Top journals by paper volume, with non-academic sector shares of all their papers:")
display(journal_company_df)



In [ ]:
h.plot_top_journal_company_share(journal_company_df.drop(columns=["share_uk_company"]), top_n=15)

## 8. Advanced Dynamics and Diagnostics

This section extends taxonomy analysis with annual diagnostics on:
- volume/share dynamics by sector,
- UK vs non-UK company composition,
- collaborator churn (new vs returning companies),
- persistence of top company collaborators over time.

In [ ]:
yearly_metrics_df = h.build_yearly_metrics_table(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
print("Yearly collaboration metrics:")
display(yearly_metrics_df)

h.plot_yearly_metrics_dashboard(yearly_metrics_df)
h.plot_company_geography_mix_over_time(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)

company_churn_df = h.build_company_collaborator_churn_table(df, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
print("Company collaborator churn (new vs returning):")
display(company_churn_df)

h.plot_new_vs_returning_company_collaborators(company_churn_df)

top_company_year_df = h.build_top_company_year_matrix(df, top_n=15, start_year=ANALYSIS_START_YEAR, end_year=ANALYSIS_END_YEAR)
print("Top company collaborators by year matrix:")
display(top_company_year_df)

h.plot_top_company_heatmap(top_company_year_df)


## 9. FoR 2020 Level-2 Analysis

FoR level-2 labels are parsed from `category_for_2020` and reported using discipline names (not only numeric codes).

In [ ]:
df, for_name_map = h.add_for_level2_columns(df)

if "category_for_2020_lvl2" in df.columns:
    n_with_lvl2 = int((df["category_for_2020_lvl2"].map(len) > 0).sum())
    unique_lvl2 = len({code for codes in df["category_for_2020_lvl2"] for code in codes})
    print(f"Papers with parsed category_for_2020 level-2 codes: {n_with_lvl2}")
    print(f"Unique parsed level-2 codes: {unique_lvl2}")
else:
    print("Column category_for_2020 not found or no level-2 parsing available.")

for_share_df = h.build_for_share_table(df, for_name_map)
if for_share_df.empty:
    print("No category_for_2020 level-2 codes found to plot.")
else:
    display(
        for_share_df[[
            "l2_for_name",
            "code",
            "papers",
            "share_hospital_clinical",
            "share_university_hei",
            "share_company_non_uk",
            "share_uk_company",
        ]]
        .sort_values("papers", ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    h.plot_for_share_table(for_share_df, top_n=15)
    h.plot_for_company_share_scatter(for_share_df, min_papers=30, max_labels=10)

uk_for_df = h.build_uk_company_for_table(df, for_name_map)
if uk_for_df.empty:
    print("No category_for_2020 level-2 codes found to plot.")
else:
    display(
        uk_for_df[["l2_for_name", "code", "papers", "uk_company_share"]]
        .sort_values("uk_company_share", ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    h.plot_uk_company_for_table(uk_for_df, top_n=15)


## 10. Geography of Collaborators

Country counts are matched from taxonomy-derived collaborator organization sets at paper level (with fallback to `research_org_country_names` where needed).

In [ ]:
df = h.add_country_columns(df)

non_country_df = h.build_country_count_table(df["non_academic_countries"])
company_country_df = h.build_country_count_table(df["company_countries"])

print("Top taxonomy-collaborator countries (all sectors combined):")
display(non_country_df.sort_values("n_pubs", ascending=False).head(20).reset_index(drop=True))

print("Top company-sector collaborator countries:")
display(company_country_df.sort_values("n_pubs", ascending=False).head(20).reset_index(drop=True))

world = h.load_world_geodata()
h.plot_country_map(non_country_df, "Taxonomy collaborator country counts", cmap_name="cividis", world=world)
h.plot_country_map(company_country_df, "Company-sector collaborator country counts", cmap_name="viridis", world=world)


## 11. Top-Cited Papers

Top-five cited papers are shown for selected taxonomy sectors,
formatted as: first author surname (`et al.` when multiple authors), year, title, journal, DOI, citation count.

In [ ]:
sections = h.build_top_cited_sections(df, top_n=5, citation_col="times_cited")

for heading, refs in sections.items():
    print()
    print(f"{heading}:")
    for i, ref in enumerate(refs, start=1):
        print(f"{i}. {ref}")


## 12. Publication Figure

A single 2x2 composite figure for publication use.

In [ ]:
saved_pubfig = h.plot_publication_figure(
    df,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    citation_col="times_cited",
    top_company_n=15,
)

---

## Execution boundary before Part 5: Assembled publication panels

The following reset deliberately reproduces the fresh Python kernel used by the former
`04_non_academic_99_all.ipynb` while leaving files written by earlier parts available to this one.


In [ ]:
# Reproduce the clean-kernel boundary that separated the source notebooks.
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass

import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()

from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part V: Assembled publication panels

**What this part is.** The one place analysis 04's figures are *selected* and
*assembled*. Parts I–IV retain the complete data work and diagnostics, while this part re-draws the charts that made the cut into
one main-paper panel and five supplementary panels.

It replaces `04_non_academic_00_all.ipynb`, which was deleted: that notebook's stored
outputs were stale against every source it read (151 clinical trials against the current
195, 348 policy documents against 449), and it drew a partial subset of the four arms
rather than assembling them.

## The argument the figures make

UK Biobank research reaches outside the academy along four routes, and the panels are
ordered to follow them:

| route | what is measured | source part |
|---|---|---|
| **patents** | patents whose front page cites a UK Biobank paper | Part II |
| **clinical trials** | registry records that cite a UK Biobank paper | Part I |
| **policy & news attention** | Altmetric news and policy mentions; Dimensions policy documents | Part III |
| **non-academic collaboration** | co-authoring organisations outside the university sector | Part IV |

**Analysis window: 1 January 2013–31 December 2025, inclusive.** Papers are filtered by publication date/year;
patents by publication date/year; clinical trials by start date; and policy documents by
year, before calculating rankings, denominators or panels. Altmetric observations are
restricted to eligible corpus papers, with attention and citation counts retained as source
snapshot totals because individual event dates are unavailable.

**Two inventories, used deliberately.** Publication reach uses the wider Dimensions
reverse index, retaining only links to outcomes with dates within the analysis window. Descriptive patent
panels use the existing detailed patent cohort. The raw endpoint snapshot and saved
classification datasets are preserved; the cutoff is applied when loading analyses.

**None of these is UK Biobank *running* a trial or filing a patent.** Every count here is
an *influence* count: someone else's patent, trial, policy document or company cites or
co-authors UK Biobank research. That framing is the point of the figure family and it
should survive into the caption.

## Re-drawn, not pasted

Parts I–IV each draw in their own figure size, type scale and palette —
the clinical-trials part uses the LCDS colours at dpi 400, the patents part
carries a `colors_scheme` literal, the Altmetric part used to set its own font. Laid
side by side those differences read as four figures stapled together. So every panel here
is re-drawn from the data through `draw_<name>(ax, D)` functions in
[`data_analysis_04_non_academic_panels.py`](../utils/data_analysis_04_non_academic_panels.py),
into a figure this part has already sized, under one style section. A colour means
the same thing in panel A as in panel F.

## The selection

Decided with the project owner on 2026-09-08. Nothing was discarded — everything not in
the main figure is in an SI panel.

| figure | panels | contents |
|---|---|---|
| **Main** | A–F | reach · cumulative growth · patents · trials · Altmetric attention · collaboration |
| **SI 1** | A–D | patents: legal status, divisions, breadth, country × division |
| **SI 1b** | — | patents: RCDC macro-cluster × category (one panel, its own page) |
| **SI 2** | A–E | trials: stage, ICD-10 body map, RCDC diseases, enrollment, country × sector |
| **SI 3** | A–F | policy: year, countries, publishers, divisions, concentration, top papers |
| **SI 4** | A–D | altmetric: score distribution, mentions by year, coverage rate, attention vs citation |
| **SI 5** | A–F | collaboration: mentions, papers, sector share, company UK/non-UK, top companies, disciplines |

Sub-panel titles are the letter and nothing else (§2 writes the captions to disk beside
the figures, so the figure and its caption cannot drift apart).

## Two conventions the panels follow

* **A bar annotated with its own value carries no grid.** The grid is there so a length can
  be estimated off the axis; once the number is printed at the end of the bar there is
  nothing left to estimate and the gridlines are only ink. Un-annotated panels keep theirs,
  dashed, on both axes and both tick levels.
* **A logarithmic axis gets major gridlines only.** A log decade holds eight minor ticks, so
  the dashed minor grid lands as a solid-looking band behind the data.
* **Heatmaps use one colormap — the palette's own cold-to-warm ramp** (navy → steel blue →
  blue → cream → red), so a reader who learns the scale in one panel keeps it in the next.
  Cells with no data are white, not navy: on a cold-to-warm ramp zero sits at the *dark*
  end, and an unmasked block-diagonal matrix renders "does not occur" in the same saturated
  navy as "is rare".
* **Type is scaled per figure** by `fs_scale` in the style section. The numbers track page
  height, because that is what sets the zoom a figure is read at: the main panel is drawn at
  1.32×, SI 2 (a page and a third tall) at 1.25×, the rest near 1.0×.

## 1. Setup

The style section is `04_non_academic_panels` — the seven-colour palette that
`shared_style.PALETTE_COLORS` names, dashed gridlines on both axes and both tick levels,
and `save: true` writing pdf + png at dpi 600 (D32) into
`output/figures/data_analysis/04_non_academic/`.

In [ ]:
import sys
from pathlib import Path

# Anchor every path on the repo root regardless of where the notebook is launched from
# (repo root, src/, or src/data_analysis/). See shared_paths for the why.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()

# The panel module is edited while this notebook stays open; without autoreload a
# re-run of a figure cell silently reuses the function imported at kernel start.
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from utils.shared_style import PALETTE_COLORS, load_style, savefig
from utils import data_analysis_04_non_academic_panels as NP

STYLE = load_style("04_non_academic_panels")

FIG_DIR = Path(STYLE["savedir"])
TABLE_DIR = P.OUTPUT_TABLES / "04_non_academic"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("palette :", ", ".join(f"{k}={v}" for k, v in PALETTE_COLORS.items()))
print("figures :", FIG_DIR, "|", ", ".join(STYLE["formats"]), "at dpi", STYLE["dpi"])
print("tables  :", P.raw_path(TABLE_DIR))
print("grid    :", STYLE["grid_linestyle"], "on both axes, major and minor")

## 2. Build the aggregates — once, from the four sources

Roughly 40 seconds, most of it the collaboration sector taxonomy (26,109 publications)
and two parquet reads. Everything the panels draw comes out of this one dict, so a figure
cannot quietly read a different vintage of the data than its neighbour.

The collaboration file is read ten columns wide rather than twenty-five: `authors` is
~95% of its 339 MB and the sector taxonomy does not use it. The taxonomy itself is
derived by the source notebook's own `add_non_academic_sector_taxonomy`.

In [ ]:
D = NP.build_panel_data()

### What the four sources actually contain

Printed rather than remembered — these are the numbers the panels are drawn on, and they
are the rows to check against `doc/STATE.md` §2 before quoting anything from the figures.

In [ ]:
print("row counts")
for key, value in D["counts"].items():
    print(f"  {key:<16} {value:>9,}")

print("\nreach — publications carrying each non-academic linkage")
display(D["reach"].assign(pct=lambda d: d["pct"].round(2)))

print("collaborator mentions by sector")
display(D["collaboration"]["sector_summary"])

## 3. The selection, on the record

One row per panel: which figure, which letter, and the caption line that goes under it.
Written to `output/tables/04_non_academic/panel_selection.csv` so the paper draft and this
notebook cannot disagree about which chart is panel D.

In [ ]:
rows = [{"figure": "main", "panel": letter, "caption": caption}
        for letter, caption in NP.MAIN_CAPTION.items()]
rows += [{"figure": f"si_{section}", "panel": letter, "caption": caption}
         for section, captions in NP.SI_CAPTIONS.items()
         for letter, caption in captions.items()]

selection = pd.DataFrame(rows)
selection.to_csv(TABLE_DIR / "panel_selection.csv", index=False)
print(f"{len(selection)} panels across {selection.figure.nunique()} figures -> "
      f"{P.raw_path(TABLE_DIR / 'panel_selection.csv')}\n")
display(selection)

## 4. The main-paper panel

Six charts on three rows, two to a row. **A** sets the scale of every stream at once; the
middle row is the translational pipeline — patents (**B**) and trials (**C**) — beside who
UK Biobank researchers actually write with (**D**); and the bottom row pairs the route that
runs through no formal artefact at all, public and policy attention (**E**), with the
*structure* of the collaboration **D** counts (**F**). That row is 1.55× the height of the
two above it, which is what both panels need: E's two named outliers sit in the empty band
above a diagonal cloud, so height is what shortens their leaders, and F is square by nature.

**The reach bars came out.** They answered "how many publications carry each linkage",
which is the same question [`01_growth`](01_growth.ipynb)'s reach panel answers on the same
numbers — two figures in one paper making one point twice. `D["reach"]` is still built, so
the counts stay quotable in the text and in `panel_selection.csv`; only the panel is gone.
The attention scatter takes the freed row across both columns, which is what lets its two
outliers be named.

Reading notes worth carrying into the caption:

* **A** is on a log axis and counts *publications*, dated by the publication's own year —
  five streams in one unit. Counting artefacts instead (patents by filing year, trials by
  start year) would put five different units on one axis.
* **B** is the *legal* outcome — what became of each patent — and it is labelled with the
  yearly total only. Seven categories in a half-width panel means most segments are a few
  pixels tall, and labelling those turns the stack into a column of overlapping digits;
  the total is the number worth reading off. The recent years leaning on "Application
  Pending" is the pipeline, not a fall: applications publish 18 months after filing, so
  2024–25 have had no time to resolve. The application-against-granted split that used to
  sit here is now SI 1 panel B.
* **F** is the sector-overlap matrix from
  Part IV, §7, re-drawn
  through the panel module in the family's own heat ramp. It is **row-normalised**: cell
  (i, j) is the share of the publications carrying sector *i* that also carry sector *j*,
  so it is asymmetric by construction and reads *across the rows*, with each row's own n on
  the tick. The asymmetry is the finding — 90.4% of UK-company papers also carry a
  university partner, against 1.4% of university papers carrying a UK company. A
  non-academic partnership sits on top of an academic one; it does not replace it. Raw
  counts would not show this at all: University/HEI carries 68,978 mentions against UK
  company's 389, so every cell would be one bright column and seven dark ones. It pays for
  the half width in type: the cells drop their percent signs — the axis label, the
  colourbar and the caption all carry the unit — the row labels put their n on a second
  line, and the column labels wrap at eleven characters.
* Both **A** and **B** carry their legends at reduced type (2 and 3 points under the family
  size). Five long stream names and a seven-category two-column legend are boxes, not
  captions, and at the family size each was wide enough to sit on the data in whichever
  corner it was put.
* **C** includes trials starting in 2013–2025, with one bar per calendar year. Earlier,
  later and undated starts are excluded before all trial summaries and rankings.
* **E** names its two most-mentioned publications. An anonymous cloud invites the reader to
  wonder which paper the top-right point is; two labels answer it. Their positions are set
  in *axes* fractions, not data coordinates, so the boxes stay in the empty band above the
  cloud whatever the axis limits do.

**Two colours, blue and red**, carry every two-category bar (**B**, **C**, and the study-type
split in SI 2). A light/dark pair of one hue needs the legend to be read at all; blue against
red separates at a glance and survives being printed small.

The figure is drawn at the `main` type scale (1.32×) — larger than the SI panels, because it
is read at figure width in a paper rather than full page on a screen. **B** and **C** are
annotated and therefore carry no grid (**C** labels every segment and every total, **B** only
the totals); **A** and **E** drop theirs too, since a log axis spanning four decades competes
with the data for the same space. **D** keeps its grid: it is the one main panel whose values
have to be read off an axis rather than from a printed number.

In [ ]:
fig_main = NP.figure_main(D)

## 5. SI 1 — Patents

What the patent arm measures beyond the main panel's single bar chart: what became of the
patents, what they are about at two levels, how broad they are, how the mix differs by
country, and how long the paper→patent step takes.

**The patent arm is two figures.** SI 1 is an ordinary 2×2 grid; the RCDC macro-cluster
heatmap has a page of its own below it. That heatmap is 28 rows of category names by seven
clusters — a shape that only reads at full page width — and forcing it into the grid had
pushed SI 1 to twice its natural height and every other panel into a squashed band. Two
figures at ordinary proportions beat one at extraordinary ones.

**SI 1b is the RCDC macro-cluster view** from §4.1 of the source notebook: the ~300 RCDC tags
partitioned into seven research programmes by Louvain community detection on their
co-occurrence graph, each column showing the four tags carrying the most patents in its
cluster. Two things follow from how it is built:

* the matrix is **block-diagonal** — a tag belongs to exactly one cluster — so it reads
  *down the columns*, not across the rows, and its row count is `top_n × 7` exactly. Four
  tags per cluster is 28 rows; five would be 35, which no panel height makes legible.
* it needs `data/analysis/non_academic/patent/paten_rcdc_macro/cluster_label_summary_louvain.csv`,
  an artefact of a job this module does not run. Without that partition the panel says so
  rather than inventing clusters, which cannot be re-derived from the patents alone.

It carries no panel letter: the figure *is* the panel, and stamping an "A" on it would only
invite the reader to look for a B.

**Two panels came out**, and their aggregates are still built if the numbers are wanted:
assignee country (`D["patents"]["countries"]` — the main figure's reach and growth panels
already carry the patent geography) and the paper→patent lag
(`D["patents"]["paper_to_patent_lag"]` — median 1 year over 800 links, which prose can carry
where a heatmap at half width cannot be read at all). Note the country count is once per
patent per country, where `doc/STATE.md` §2's "top US 268" row counts assignee
*occurrences* — different quantities off the same column.

In [ ]:
fig_si1 = NP.figure_si_patents(D)


## 6. SI 2 — Clinical trials

Drawn in navy throughout. The arm's own notebook uses LCDS blue `#344874`; navy `#274668`
is the project palette's nearest, so the clinical-trials panels keep the identity they had.

**Panel B is a body map**, not a ranked bar: trials per ICD-10 chapter, placed where the
chapter sits anatomically, with dot area proportional to the trial count. A ranked bar of
condition names answers "which conditions", and panel C already answers that with RCDC —
the body map answers the question a bar cannot, which is *where* the trials that lean on
UK Biobank concentrate. The silhouette is the artwork under
`output/figures/data_analysis/04_non_academic/clinical_trials/human_silhouette/`.

Three things to keep straight when reading it:

* **A trial counts once per chapter it touches**, so the dots do not partition the 195 — a
  trial studying both obesity and heart failure appears in IV and in IX.
* **Neoplasms and infectious disease sit off the body**, under "Systemic". They belong to no
  organ, and placing them on one would be an anatomical claim the data does not make.
* Chapters come from keyword rules over the shipped `mesh_terms` column — 162 of 195 trials
  map to at least one. Panel C asks the same question of RCDC, with the cross-cutting
  research-area tags stop-listed (D11).

**Panel E answers "who runs them", not just "where".** Each trial spreads a weight of 1
evenly across its distinct (country, sector) organisation pairs, so the bars sum to the
number of located trials rather than counting a multi-site trial once per site — the same
fractional accounting as §7 of the source notebook, and the reason the totals print as
whole numbers while the segments behind them are not integers. Sector comes from the
organisation's GRID type where it is committal, and from its name where it is not
(`Facility`, `Other`, untyped), which is how a teaching hospital lands in Healthcare rather
than Academia.

The paper-to-trial lag panel came out to give the body map its two rows. Its distribution
is in `D["trials"]["paper_to_trial_lag"]` — median 3 years over 213 links, with a long
negative tail that is real rather than an error: a trial that started in 1996 and cites a
2023 paper produces a lag of −27 years.

In [ ]:
fig_si2 = NP.figure_si_trials(D)

## 7. SI 3 — Policy documents

This is the **Dimensions** policy-citation route (449 documents, 369 publications cited).
It is *not* the same quantity as Altmetric's policy mentions in SI 4 (579 publications) —
the two sources index different document sets and D18 records that they are not
reconcilable. Neither is wrong; they answer slightly different questions and the panels
keep them apart.

**Four panels, drawn in the palette's blue.** The stream palette assigns policy the
palette's red, which is what the main figure's five-line panel A needs to tell five streams
apart. This page is about one stream, so a second hue would carry no information and it is
drawn end to end in `POLICY_PRIMARY`.

**B is now a choropleth.** The ranked country bar's finding was never which country comes
first — it is that a UK resource is cited by policy bodies on five continents, and that the
second and third publishers (the United States and Switzerland, the latter almost entirely
the WHO) are not British. A bar makes the reader assemble that geography from twelve country
names; a map states it. The colour scale is logarithmic, because the counts run 1 to 111
with most countries in single figures and a linear ramp would paint everything outside the
top three the same near-white. Countries with no citing document are grey, not the ramp's
lightest blue — absent and minimal are different claims. The leading five are named with
their counts under the map, since a choropleth cannot be read to a number.
`draw_policy_countries` still draws the bar if it is wanted.

**Two panels came out.** The concentration histogram and the ranked list of most-cited
papers were the page's two weakest claims: the first is a one-line fact — median 1, max 22
over 369 publications — that prose carries better than a log-axis histogram of a
distribution with nothing in its tail, and the second is eight truncated paper titles, which
is a table pretending to be a chart. Both aggregates are still built
(`D["policy"]["policy_docs_per_paper"]`, `D["policy"]["top_cited_papers"]`) and both draw
functions are still in the module.


In [ ]:
fig_si3 = NP.figure_si_policy(D)

## 8. SI 4 — Altmetric attention

The real Altmetric Explorer export (D18), so panel B's news series is a measurement rather
than the zeros the rebuilt-from-corpus substitute would supply.

**One blue family, not one hue per panel.** The page used to rotate the palette across its
panels — light blue for the score, red for the policy series, green for the
attention-against-citation scatter — which reads as four unrelated charts.
`ATTENTION_PRIMARY` now carries every panel and `ATTENTION_SECONDARY` (navy) appears only in
B and C, where a panel genuinely draws two series and the contrast is doing work. The dashed
red median rule in A is the family's reference colour (`REF_MEDIAN`), not a topic colour,
and stays.

Panel C is a rate, not a total, and it falls across the window. That is mostly an
accumulation artefact — a 2015 paper has had ten years to be written about and a 2025
paper has had months — so it should not be read as declining public interest.


In [ ]:
fig_si4 = NP.figure_si_altmetric(D)

## 9. SI 5 — Non-academic collaboration

Panel A's counts are the ones in `doc/STATE.md` §2: University/HEI 68,978 · Hospital/
Clinical 33,873 · Research institute 8,437 · Government 4,491 · Nonprofit 4,233 · Company
non-UK 2,207 · UK company 389 · Other 101.

**The UK-company series carries a known caveat (D27).** After the GRID country correction,
4 organisations — about 15 of the 389 UK-company mentions, 3.9% — are UK bodies that are
not companies. The series is a slight over-count and is the smallest in the panel either
way.

In [ ]:
fig_si5 = NP.figure_si_collaboration(D)

## 10. What was written

Every publication file this part produced, with its size, so a re-run can be compared against the
last one rather than trusted.

In [ ]:
written = sorted(
    p for p in FIG_DIR.glob("04_[0-9][0-9]_*.*")
    if p.suffix.lower() in {".pdf", ".png"}
)
manifest = pd.DataFrame([
    {"file": P.raw_path(p), "kb": p.stat().st_size // 1024,
     "modified": pd.Timestamp(p.stat().st_mtime, unit="s").round("s")}
    for p in written
])
manifest.to_csv(TABLE_DIR / "panel_manifest.csv", index=False)
print(f"{len(written)} figure files under {P.raw_path(FIG_DIR)} "
      f"({manifest.kb.sum() / 1024:.1f} MB)\n")
display(manifest)